# Python prompting on the situational test cases — multi-model study

**Research question.** Can an LLM take a natural-language description of a realistic
situation and construct a formal POSG world model that a safe-by-design planning
algorithm could later use?

The LLM is a **neutral world modeller**, not the agent making the decision.

## What this notebook runs

The grid is **11 situational cases × 5 models × k generations**, on one prompt strategy,
plus up to three mechanical repair rounds on every first-pass failure. Repairs are always
reported separately from first passes.

| Model | Backend | Reached via |
|---|---|---|
| Qwen3.8 27B | `local_hf` | weights downloaded and generated on this notebook's own GPU |
| DeepSeek V4 Flash | `openrouter` | OpenRouter |
| GPT-5.6 Sol | `openrouter` | OpenRouter |
| Claude Opus 5 | `openrouter` | OpenRouter |
| Llama 3.3 70B | `openrouter` | OpenRouter |



**One prompt strategy, `v2_python`** — the JSON track's `zero_shot_v2` prompt with its
schema section replaced by Jobst's ABC.


## Scope

Only repository files matching
`data/Test_Cases/02_situational/situational_S_<n>_Test_Case_S<n>.md`.
Game-track and cultural-track cases are not loaded.



## Three levels of correctness, and how they map onto the JSON track

The JSON-prompting track scores models in three lanes against an error taxonomy. The
Python track measures the same ladder on a different output format, so the two can be
reported side by side:

| This notebook | Decided by | JSON track equivalent |
|---|---|---|
| **Python validity** — parses, imports, instantiates, defines exactly one POSG subclass | machine | Lane 1, taxonomy Cat 1 (invalid output) |
| **Mechanical POSG validity** — state shapes, action sets, distributions summing to 1, observation profiles, terminal mechanics | machine | Lane 1, taxonomy Cat 2–3 (schema-invalid / mechanically invalid) |
| **Semantic world-model quality** — is this a sensible model of *this* situation? | human review support only | Lane 3 (coverage / bias / human review) |

Lane 2 (structural comparison against a ground-truth reference) has **no equivalent
here**: the situational cases have no ground truth by construction. That is the point of
using them.

**Mechanical validity is never treated as semantic correctness.** A model can pass every
check in this notebook while omitting the most important stakeholder in the situation.


## A. Colab setup, private-repository read, and Drive-only output

Before running in Colab:

1. A GPU is **not** required for the hosted-router backend. Enable one only if you also
   want to run the `local_hf` backend.
2. In Colab **Secrets**, create and share with this notebook:
   - `GITHUB_TOKEN` — read access to the source repository.
   - `HF_TOKEN` — a Hugging Face token with the **"Make calls to Inference Providers"**
     permission. Without that permission the router rejects every request.
3. Run the notebook and authorize the Google Drive mount.

Both tokens should be read-only/minimal. They are used only by the clone subprocess and
the router client, and are never passed to generated-code subprocesses.

> **Cost warning.** The default grid is 11 × 5 × 5 × 3 = **825 first-pass generations**
> plus repairs, against five hosted models. Run section B's preflight and the
> `SMOKE_TEST_ONLY` pass first; only then set `SMOKE_TEST_ONLY = False`.


In [ ]:
# --- environment -------------------------------------------------------------------
import ast
import datetime as dt
import hashlib
import importlib.metadata as importlib_metadata
import itertools
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import types
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    raise RuntimeError(
        "This notebook is intentionally configured for Google Colab because all "
        "experiment outputs must be written to Google Drive."
    )

# Mount Drive BEFORE creating any experiment output directory.
drive.mount('/content/drive')

# `openai` drives both hosted backends (Azure Foundry and the HF router);
# torch/transformers load the local model; networkx backs the graph EDA.
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'openai', 'pandas', 'matplotlib', 'networkx', 'huggingface_hub',
        'transformers', 'accelerate', 'bitsandbytes',
    ],
    check=True,
)

import matplotlib
import pandas as pd
from IPython.display import Markdown, display

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else None
    GPU_GB = (
        round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
        if HAS_GPU else 0
    )
except Exception:
    torch, HAS_GPU, GPU_NAME, GPU_GB = None, False, None, 0

print('GPU:', GPU_NAME or 'none -- required for the local model')
if HAS_GPU:
    print(f'GPU memory: {GPU_GB} GB')


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.3 GB


In [ ]:
# ============================ EDIT ME ================================================
# Source repository: INPUT ONLY. Must contain
#   data/Test_Cases/02_situational/   (the cases)
#   python_prompting/                 (the prompt, analysis, figure and model code)
#   world_model_eda/                  (Gagana's graph EDA, shared with the game track)
#   posg.py                           (Jobst's ABC — the target interface)
# Point this at whichever fork holds your current work.
REPO_URL = 'https://github.com/AmnaJamalKhattak/world-model.git'
REPO_BRANCH = None              # None = the repository's default branch
GITHUB_SECRET_NAME = 'github-token'

# ALL experiment results go here on Google Drive.
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/situational_posg_runs')

# ---- which models -------------------------------------------------------------------
# The backend is declared PER MODEL in python_prompting/models.py, so one grid holds the
# locally loaded model and the two Azure deployments together: same cases, same prompt,
# same figures, one comparison.
#
#   'DEFAULT_MODELS' -> Qwen3.8 27B on this GPU, and DeepSeek V4 Flash, GPT-5.6 Sol,
#                       Claude Opus 5 and Llama 3.3 70B on OpenRouter
#   'AZURE_MODELS'   -> the two Azure AI Foundry deployments, for when they exist
#   'ROUTER_MODELS'  -> the earlier five hosted models, via the HF Inference router
REGISTRY = 'DEFAULT_MODELS'
MODEL_FILTER = ()               # e.g. ('qwen3.8-27b',) to run one while iterating

# ---- local weights (models whose backend is 'local_hf') -----------------------------
# LOCAL_MODEL_ID overrides the registry's repo id. Set it if the preflight below cannot
# resolve the declared one: it prints the real repository names to choose from.
# LEAVE IT EMPTY to use whatever the registry declares.
LOCAL_MODEL_ID = ''
# 'none' loads in bfloat16: right for a 27B model on an 80 GB A100 (~54 GB of weights).
# '8bit' / '4bit' fit a smaller card but CHANGE THE MODEL'S OUTPUTS, so a quantised row
# is not comparable with an unquantised one. Whatever is used is written to config.json.
LOCAL_QUANTIZATION = 'none'
HF_SECRET_NAME = 'HF_TOKEN'     # needed to download a gated repository

# ---- Azure AI Foundry (models whose backend is 'azure') -----------------------------
# UNUSED by DEFAULT_MODELS: the hosted models moved to OpenRouter because a Foundry model
# is not callable until it is deployed. Kept working for REGISTRY = 'AZURE_MODELS', which
# needs deployments to exist first (Foundry -> Models + endpoints -> + Deploy model ->
# Global Standard, never Provisioned/PTU, which bills hourly whether you call it or not).
# The endpoint from the Foundry portal. The '/openai/v1' suffix is added if missing.
AZURE_ENDPOINT = ''
AZURE_AUTH = 'api_key'          # 'api_key' (a Colab secret) or 'entra_id' (az login)
AZURE_SECRET_NAME = 'AZURE_API_KEY'
# Deployment names as they appear in YOUR Foundry project. The preflight lists what the
# endpoint actually serves, so these can be corrected before anything is generated.
AZURE_DEPLOYMENTS = {
    'deepseek-v4-flash': 'DeepSeek-V4-Flash',
    'gpt-5.6-sol': 'gpt-5.6-sol',
}

# ---- OpenRouter (models whose backend is 'openrouter') ------------------------------
# One key, many vendors, one OpenAI-compatible endpoint. Unlike Azure, its catalogue
# lists models that are actually CALLABLE, so the preflight below is a real check.
OPENROUTER_SECRET_NAME = 'open-router'
# Model ids as OpenRouter lists them ('vendor/model'). The preflight verifies these
# against the live catalogue and, on a miss, prints that vendor's real ids.
OPENROUTER_MODELS = {
    'deepseek-v4-flash': 'deepseek/deepseek-v4-flash',
    'gpt-5.6-sol':       'openai/gpt-5.6-sol',
    'claude-opus-5':     'anthropic/claude-opus-5',
    'llama-3.3-70b':     'meta-llama/llama-3.3-70b-instruct',
}
# Optional: pin the upstream provider per model, e.g. {'claude-opus-5': 'Anthropic'}.
# OpenRouter fans one model id out to several providers that need not serve the same
# quantisation, so an unpinned run is not exactly reproducible. Whichever provider
# served each call is recorded in the attempt either way.
OPENROUTER_PROVIDER_PIN = {}
# Attribution headers, shown on the OpenRouter dashboard. Cosmetic.
OPENROUTER_TITLE = 'world models'

# ---- Azure spending guard -----------------------------------------------------------
# The cost probe sends one or two ten-token calls per deployment and reports what they
# cost IN TOKENS. It CANNOT tell you who pays: no API response names the funding source,
# so promotional credit and a card look identical from here.
#
# Read it in the portal:  Azure Portal -> Cost Management + Billing -> your subscription
#                         -> Credits (promotional balance) and Payment methods.
#
# Until AZURE_BILLING_CHECKED is True, the run loop REFUSES to call any Azure model and
# the grid runs the local model only. Set it to True yourself, after you have looked.
RUN_COST_PROBE = True
COST_PROBE_CALLS = 2
AZURE_BILLING_CHECKED = False

# ---- experiment shape ---------------------------------------------------------------
# One prompt strategy, `v2_python`: the JSON track's zero_shot_v2 prompt with its schema
# section replaced by Jobst's ABC. See python_prompting/prompts.py for the four clauses
# that had to change and why. Running one strategy is what makes the two tracks'
# numbers comparable.
N_GENERATIONS = 3              # k, for pass@k and self-consistency
CASE_FILTER = ()               # e.g. ('S.4', 'S.10'); empty = all repo situational cases
ENABLE_REPAIR = True
# Text2World (arXiv 2502.13092) reports executability rising from 48.5% to 72.3% over
# THREE correction attempts, making iterative repair the largest single lever they found.
# Repairs are always reported separately from first passes, so this never inflates the
# headline rate -- it increases how many usable models reach the handoff.
MAX_REPAIR_ROUNDS = 3
RESUME_RUN_ID = '20260829-091634_default_models'          # exact existing Drive run ID, or None for a new run

# Start here. One case x one generation on every model, to prove the wiring and see
# per-model latency and token cost before committing to the full grid.
SMOKE_TEST_ONLY = False

# Decoding.
TEMPERATURE = 0.2
TOP_P = 0.95
REPETITION_PENALTY = 1.05      # local_hf only; hosted backends ignore it
# A POSG class for a real situation runs to a few hundred lines, and a reasoning model
# spends its budget thinking BEFORE it writes any of it. At 2048 a reasoning model can
# return nothing at all -- not a refusal, just an unfinished thought. 4096 leaves room
# for both; `truncated` in every record says whether even that ran out.
OPENROUTER_REASONING_DEFAULT = 'max_tokens:4096'   # new line
OPENROUTER_REASONING = {}
MAX_NEW_TOKENS = 16384
SEED = 0

# Backend robustness.
MAX_ATTEMPTS = 5               # retries per call, on rate limits and 5xx only
REQUEST_TIMEOUT_S = 600.0

# Analysis.
GRAPH_ALL_ATTEMPTS = True     # True draws a graph for every valid attempt, not one per (case, model)
RUN_WORLD_MODEL_EDA = True     # Gagana's graph EDA panels, on top of the 16 figures
EDA_ALL_ATTEMPTS = False       # as above, for the EDA panels
BOOTSTRAP_DRAWS = 2000         # cluster bootstrap resamples for paired comparisons
CONFIDENCE = 0.95

# Mechanical validation. Hitting a cap => INCOMPLETE, never PASS.
PROBE_TIMEOUT_S = 60
CAPS = {
    'max_states': 400,
    'max_profiles': 200,
    'max_depth': 12,
    'max_samples': 8,
    'max_transition_samples': 50,
    'max_errors': 25,
}
# =====================================================================================

# Clone the source repository for READ-ONLY INPUT.
SOURCE_CLONE_DIR = Path('/content/world-model-source')
if SOURCE_CLONE_DIR.exists():
    shutil.rmtree(SOURCE_CLONE_DIR)

github_token = userdata.get(GITHUB_SECRET_NAME)
if not github_token:
    raise RuntimeError(
        f"Colab secret {GITHUB_SECRET_NAME!r} is missing or is not shared with this notebook."
    )

askpass = Path('/tmp/world_model_git_askpass.sh')
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
    'esac\n',
    encoding='utf-8',
)
askpass.chmod(0o700)

clone_env = {
    'PATH': os.environ.get('PATH', ''),
    'HOME': os.environ.get('HOME', '/root'),
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GITHUB_TOKEN': github_token,
}

try:
    command = ['git', 'clone', '--quiet']
    if REPO_BRANCH:
        command += ['--branch', REPO_BRANCH]
    command += [REPO_URL, str(SOURCE_CLONE_DIR)]
    proc = subprocess.run(command, env=clone_env, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError('Repository clone failed.\n' + (proc.stderr or proc.stdout)[-2000:])
finally:
    askpass.unlink(missing_ok=True)
    del clone_env
    del github_token

REPO_ROOT = SOURCE_CLONE_DIR.resolve()
SOURCE_COMMIT_SHA = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True
).strip()
SOURCE_BRANCH = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'branch', '--show-current'], text=True
).strip()

# Remove Git metadata so this working tree cannot be accidentally committed/pushed.
GIT_DIR = REPO_ROOT / '.git'
if not GIT_DIR.is_dir():
    raise RuntimeError('Expected .git after clone, but it was not found.')
shutil.rmtree(GIT_DIR)
if GIT_DIR.exists():
    raise RuntimeError('Failed to remove .git from the source clone.')

POSG_ABC_PATH = REPO_ROOT / 'posg.py'
SITUATIONAL_DIR = REPO_ROOT / 'data' / 'Test_Cases' / '02_situational'
PACKAGE_DIR = REPO_ROOT / 'python_prompting'
EDA_PACKAGE_DIR = REPO_ROOT / 'world_model_eda'
for required, what in (
    (POSG_ABC_PATH, "Jobst's POSG ABC (posg.py)"),
    (SITUATIONAL_DIR, 'situational test-case directory'),
    (PACKAGE_DIR, 'python_prompting analysis package'),
    (EDA_PACKAGE_DIR, 'world_model_eda graph analysis package'),
):
    if not required.exists():
        raise FileNotFoundError(
            f'{what} not found in the clone: {required}\n'
            f'Check REPO_URL / REPO_BRANCH — this notebook needs a repository that has '
            f'the situational cases, posg.py, python_prompting and world_model_eda.'
        )

# Import the analysis/figure/model code FROM THE PINNED CLONE, so the numbers in the
# paper are tied to SOURCE_COMMIT_SHA rather than to whatever is pasted in a cell.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from python_prompting import analysis as ANALYSIS      # noqa: E402
from python_prompting import figures as FIGURES        # noqa: E402
from python_prompting import models as MODELREG        # noqa: E402
from python_prompting import tables as TABLES          # noqa: E402

# Exact situational filename convention supplied by the project.
CASE_FILE_RE = re.compile(r'^situational_S_(\d+)_Test_Case_S(\d+)\.md$')
SITUATIONAL_FILES = []
for path in SITUATIONAL_DIR.glob('situational_S_*_Test_Case_S*.md'):
    match = CASE_FILE_RE.match(path.name)
    if not match:
        continue
    if match.group(1) != match.group(2):
        raise ValueError(f'Filename contains inconsistent case numbers: {path.name}')
    SITUATIONAL_FILES.append(path)
SITUATIONAL_FILES.sort(key=lambda p: int(CASE_FILE_RE.match(p.name).group(1)))
if not SITUATIONAL_FILES:
    raise FileNotFoundError(
        'No files matched data/Test_Cases/02_situational/situational_S_*_Test_Case_S*.md'
    )

# Drive run directory.
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = DRIVE_OUTPUT_ROOT / RUN_ID
    if not RUN_DIR.is_dir():
        raise FileNotFoundError(f'RESUME_RUN_ID does not exist on Drive: {RUN_DIR}')
else:
    RUN_ID = f"{dt.datetime.now():%Y%m%d-%H%M%S}_{REGISTRY.lower()}"
    RUN_DIR = DRIVE_OUTPUT_ROOT / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)

# ---- the Drive layout ----------------------------------------------------------------
# One mapping, used by every cell that writes anything, so the folder names on Drive say
# what they hold without anyone having to ask. Numbered in the order the pipeline fills
# them, so a teammate opening the run directory reads it top to bottom.
RUN_LAYOUT = {
    'info':       '00_run_info',
    'cases':      '01_inputs/situation_cases',
    'prompts':    '01_inputs/prompts_sent',
    'raw':        '02_model_outputs/raw_replies',
    'code':       '02_model_outputs/generated_python',
    'probe':      '03_validation/probe_reports',
    'selftest':   '03_validation/harness_selftests',
    'first_pass': '04_handoff_to_evaluation/first_pass_v2_json',
    'repaired':   '04_handoff_to_evaluation/repaired_v2_json',
    'derived':    '05_analysis_tables/derived_tables',
    'latex':      '05_analysis_tables/latex_tables',
    'figures':    '06_figures',
    'graphs':     '07_state_graphs',
    'eda':        '08_graph_eda',
    'review':     '09_human_review_sheets',
}


def folder(key: str) -> Path:
    """The Drive directory for one kind of artefact. Created by the loop below."""
    return RUN_DIR / RUN_LAYOUT[key]


for relative in RUN_LAYOUT.values():
    (RUN_DIR / relative).mkdir(parents=True, exist_ok=True)

RECORDS_PATH = folder('info') / 'records.jsonl'

# A plain-language map of the run directory, written before anything else, so whoever
# opens the folder on Drive does not have to reverse-engineer it from filenames.
(RUN_DIR / '00_START_HERE.md').write_text(f"""# Situational POSG world models — run `{RUN_ID}`

LLMs were asked to turn each of the 11 situational test cases (S.1-S.11) into a formal
POSG world model written as Python, implementing `posg.py` from the project repository.

Source commit: `{SOURCE_COMMIT_SHA}`
Created: {dt.datetime.now():%Y-%m-%d %H:%M}

## What is in each folder

| Folder | What it holds |
|---|---|
| `00_run_info/` | What was run and how. `config.json` (every setting), `provenance.json` (commit + file hashes), `records.jsonl` (one record per attempt — the input to every number below, and the resume cache), `preflight_models.csv`, `cost_probe.csv`, `prompt_divergences.csv`, `manifest.tsv` |
| `01_inputs/situation_cases/` | The exact case files the models were shown, copied from the repository |
| `01_inputs/prompts_sent/` | The exact prompt sent for every attempt, plus `v2_python.system.md` — the one system prompt this study uses |
| `02_model_outputs/raw_replies/` | Untouched replies. Nothing stripped, nothing repaired |
| `02_model_outputs/generated_python/` | The Python extracted from each reply and actually executed |
| `03_validation/probe_reports/` | Full mechanical validation output per attempt: what passed, what failed, and why |
| `03_validation/harness_selftests/` | Evidence the checker itself works — 18 deliberately broken models, all rejected |
| **`04_handoff_to_evaluation/first_pass_v2_json/`** | **THE DELIVERABLE.** Every mechanically valid model the LLMs produced unaided, in the v2 JSON schema `json_generation/lane1.py` reads |
| `04_handoff_to_evaluation/repaired_v2_json/` | Models that became valid only after our harness fed back their errors. A DIFFERENT object — never mix these with the first passes |
| `05_analysis_tables/` | Our own analysis: tidy CSVs and booktabs LaTeX tables |
| `06_figures/` | Publication figures, PNG and vector PDF |
| `07_state_graphs/` | The state graph of each generated model, plus one per-case panel comparing every model on the same situation |
| `08_graph_eda/` | The same graph statistics the game track reports, via `world_model_eda` |
| `09_human_review_sheets/` | One sheet per first-pass generation, for semantic review. Nothing here is auto-scored |

## What to send for evaluation

`04_handoff_to_evaluation/first_pass_v2_json/` and its `handoff_lane1_check.csv`.

Send `repaired_v2_json/` only if it is clearly labelled as repaired. A first pass is what
the LLM produced; a repaired model is what it produced after our harness told it what was
wrong. The JSON track does not repair at all, so only first passes are comparable across
the two tracks.

## What these numbers do and do not say

Everything machine-checked here is **mechanical validity**: the model parses, runs,
implements the interface, and its distributions sum to 1. That is Lane 1.

It is never evidence of **semantic** quality. A model can pass every check in this run
and still leave out the most important person in the situation. That is what
`09_human_review_sheets/` is for, and it needs a human.
""", encoding='utf-8')

if not str(RUN_DIR).startswith('/content/drive/'):
    raise RuntimeError(f'RUN_DIR is not on Google Drive: {RUN_DIR}')
if str(RUN_DIR).startswith(str(REPO_ROOT)):
    raise RuntimeError('RUN_DIR must never be inside the source repository.')
if (REPO_ROOT / '.git').exists():
    raise RuntimeError('Source clone unexpectedly still contains .git.')

print('source commit:', SOURCE_COMMIT_SHA)
print('source branch:', SOURCE_BRANCH or '(detached)')
print('source .git removed:', not (REPO_ROOT / '.git').exists())
print('situational cases found:', len(SITUATIONAL_FILES))
print('Drive run directory:', RUN_DIR)


source commit: a6e7fd05f8d492e04e8e8361fd78e4fa7bb455fd
source branch: main
source .git removed: True
situational cases found: 11
Drive run directory: /content/drive/MyDrive/situational_posg_runs/20260829-091634_default_models


### A.1 Model registry and per-backend preflight

Models are declared in `python_prompting/models.py`, version-pinned with the rest of the
analysis code by `SOURCE_COMMIT_SHA`. Each one carries its **own backend**, so this run
holds a locally loaded model and two Azure deployments in one grid — same cases, same
prompt, same figures.

| model | backend | how it is reached |
|---|---|---|
| Qwen3.8 27B | `local_hf` | weights downloaded and generated on this notebook's GPU |
| DeepSeek V4 Flash | `openrouter` | OpenRouter |
| GPT-5.6 Sol | `openrouter` | OpenRouter |
| Claude Opus 5 | `openrouter` | OpenRouter |
| Llama 3.3 70B | `openrouter` | OpenRouter |

The preflight below checks each of them against the service that will actually serve it,
**before** any generation happens:

- **local** — that the repository resolves on the Hub, and that this GPU has room for it.
  If the id does not resolve the cell **fails and lists real repository names**. It never
  substitutes a neighbouring checkpoint: publishing one model's results under another
  model's name is the worst outcome available here, and it is silent.
- **OpenRouter** — that each model id is in the live catalogue, which lists models that
  are genuinely callable. A miss fails the cell with that vendor's real ids. It also
  pulls each model's listed price, so A.2 can quote what the grid will cost, and reports
  the key's remaining prepaid credit where the key exposes it.

**Azure is no longer in the default grid.** Its backend still works
(`REGISTRY = 'AZURE_MODELS'`) but a Foundry model is not callable until somebody deploys
it: `/models` there returns the **catalogue** of what you *may* deploy, so a name
appearing in it is not evidence a call will work, and an undeployed entry answers a
perfectly valid key with 404 `DeploymentNotFound`. That is why these two models moved.

**Azure spending guard.** Until `AZURE_BILLING_CHECKED = True`, the Azure models are held
back and the grid runs the local model only. The cost probe two cells down sends one or
two tiny calls so you can see the deployments answer and what they cost in tokens; then
you check the funding source in the portal, set the flag, and re-run cells 3 and 5.


In [ ]:
from python_prompting import prompts as PROMPTS  # noqa: E402

REGISTRY_SPECS = {
    'DEFAULT_MODELS': MODELREG.DEFAULT_MODELS,
    'ROUTER_MODELS': MODELREG.ROUTER_MODELS,
}
if REGISTRY not in REGISTRY_SPECS:
    raise ValueError(f'REGISTRY must be one of {sorted(REGISTRY_SPECS)}, not {REGISTRY!r}')
MODEL_SPECS = REGISTRY_SPECS[REGISTRY]

if MODEL_FILTER:
    unknown = sorted(set(MODEL_FILTER) - {s.slug for s in MODEL_SPECS})
    if unknown:
        raise ValueError(f'Unknown MODEL_FILTER entries: {unknown}')
    MODEL_SPECS = tuple(s for s in MODEL_SPECS if s.slug in MODEL_FILTER)

# Apply the cell-3 overrides to the registry's declarations.
import dataclasses as _dc  # noqa: E402

def _apply_overrides(spec):
    if spec.backend == 'local_hf':
        changes = {'quantization': LOCAL_QUANTIZATION}
        if LOCAL_MODEL_ID:
            changes['repo_id'] = LOCAL_MODEL_ID
        return _dc.replace(spec, **changes)
    if spec.backend == 'azure' and spec.slug in AZURE_DEPLOYMENTS:
        return _dc.replace(spec, deployment=AZURE_DEPLOYMENTS[spec.slug])
    if spec.backend == 'openrouter':
        changes = {}
        if spec.slug in OPENROUTER_MODELS:
            changes['repo_id'] = OPENROUTER_MODELS[spec.slug]
        if spec.slug in OPENROUTER_PROVIDER_PIN:
            changes['provider'] = OPENROUTER_PROVIDER_PIN[spec.slug]
        return _dc.replace(spec, **changes) if changes else spec
    return spec

MODEL_SPECS = tuple(_apply_overrides(spec) for spec in MODEL_SPECS)
BACKENDS_IN_USE = sorted({spec.backend for spec in MODEL_SPECS})
print('backends in this run:', ', '.join(BACKENDS_IN_USE))

preflight_rows = []
blocked: dict[str, str] = {}          # slug -> why it cannot run
HF_TOKEN = None
AZURE_CLIENT = None
AZURE_SERVED = None

# ---- local weights ------------------------------------------------------------------
LOCAL_SPECS = [s for s in MODEL_SPECS if s.backend == 'local_hf']
if LOCAL_SPECS:
    if not HAS_GPU:
        raise RuntimeError(
            'A local model is in the registry but this runtime has no GPU. '
            'Runtime -> Change runtime type -> A100 GPU, then re-run from cell 2.'
        )
    HF_TOKEN = userdata.get(HF_SECRET_NAME)      # None is fine for an ungated repo
    print(f'GPU: {GPU_NAME} ({GPU_GB} GB)')
    for spec in LOCAL_SPECS:
        found = MODELREG.verify_local_model(spec.repo_id, HF_TOKEN)
        if not found['ok']:
            listing = '\n  '.join(found['suggestions'][:20]) or '(the Hub search returned nothing)'
            raise RuntimeError(
                f"Could not resolve {spec.repo_id!r} on the Hugging Face Hub.\n"
                f"  {found['error']}\n\n"
                f"Repositories matching that name:\n  {listing}\n\n"
                f"Set LOCAL_MODEL_ID in cell 3 to the exact one you mean and re-run.\n"
                f"Nothing is guessed here on purpose: substituting a neighbouring "
                f"checkpoint would publish its results under {spec.label!r}."
            )
        print(f'  {spec.repo_id} resolved at revision {found["sha"]}')
        # Roughly 2 bytes per parameter in bfloat16, plus activations and KV cache.
        needed = (spec.params_b or 0) * 2 * 1.15
        if spec.quantization == 'none' and needed and GPU_GB < needed:
            print(f'  WARNING: {spec.label} needs about {needed:.0f} GB unquantised and '
                  f'this GPU has {GPU_GB} GB. Set LOCAL_QUANTIZATION to "8bit" or "4bit" '
                  f'in cell 3 -- and say so in the paper, because it changes the outputs.')

# ---- Azure AI Foundry ---------------------------------------------------------------
AZURE_SPECS = [s for s in MODEL_SPECS if s.backend == 'azure']
if AZURE_SPECS:
    if not AZURE_ENDPOINT:
        raise RuntimeError(
            'Azure models are in the registry but AZURE_ENDPOINT is empty in cell 3.\n'
            'Copy it from the Azure AI Foundry portal (Overview -> Endpoints).'
        )
    azure_key = userdata.get(AZURE_SECRET_NAME) if AZURE_AUTH == 'api_key' else None
    if AZURE_AUTH == 'api_key' and not azure_key:
        raise RuntimeError(
            f'Colab secret {AZURE_SECRET_NAME!r} is missing or is not shared with this '
            f'notebook. Add the Foundry key there, or set AZURE_AUTH to "entra_id".'
        )
    AZURE_CLIENT = MODELREG.build_azure_client(
        AZURE_ENDPOINT,
        MODELREG.azure_credential(AZURE_AUTH, api_key=azure_key),
        timeout=REQUEST_TIMEOUT_S,
    )
    del azure_key
    AZURE_SERVED = MODELREG.fetch_azure_deployments(AZURE_CLIENT)
    if AZURE_SERVED is None:
        print('  endpoint does not expose a model list; deployment names cannot be '
              'checked up front and will be verified by the cost probe instead')
    else:
        # NOTE: this is Azure's model CATALOGUE, not the deployments on the resource.
        # Absence proves a call will fail; presence proves nothing. Only the cost probe
        # in A.2 can confirm a deployment exists.
        print(f"  endpoint's catalogue lists {len(AZURE_SERVED)} model(s) -- what you "
              f"MAY deploy, not what IS deployed")
        for spec in AZURE_SPECS:
            if spec.deployment not in AZURE_SERVED:
                blocked[spec.slug] = (
                    f"{spec.deployment!r} is not even in this endpoint's catalogue, so "
                    f"it cannot be deployed here under that name")
        in_catalogue = [s.deployment for s in AZURE_SPECS if s.slug not in blocked]
        if in_catalogue:
            print(f'  in the catalogue: {", ".join(in_catalogue)} -- the cost probe in '
                  f'A.2 checks whether they are actually DEPLOYED')

    # The spending guard. Held-back models still reach the cost probe below; what they
    # cannot do is run the grid.
    if not AZURE_BILLING_CHECKED:
        for spec in AZURE_SPECS:
            blocked.setdefault(
                spec.slug,
                'held back: AZURE_BILLING_CHECKED is False. Run the cost probe, check '
                'Azure Portal -> Cost Management + Billing, then set the flag in cell 3.')

# ---- OpenRouter ---------------------------------------------------------------------
OPENROUTER_SPECS = [s for s in MODEL_SPECS if s.backend == 'openrouter']
OPENROUTER_CLIENT = None
OPENROUTER_SERVED = None
OPENROUTER_CATALOGUE = None
OPENROUTER_CREDITS = None
if OPENROUTER_SPECS:
    openrouter_key = userdata.get(OPENROUTER_SECRET_NAME)
    if not openrouter_key:
        raise RuntimeError(
            f'Colab secret {OPENROUTER_SECRET_NAME!r} is missing or is not shared with '
            f'this notebook. Create it at openrouter.ai/keys.'
        )
    OPENROUTER_CLIENT = MODELREG.build_openrouter_client(
        openrouter_key, timeout=REQUEST_TIMEOUT_S, title=OPENROUTER_TITLE)

    # This catalogue lists models that are actually callable, so unlike the Azure one a
    # hit here is real evidence. It also carries each model's price, which is what lets
    # A.2 quote a dollar ceiling for the grid before anything is generated.
    OPENROUTER_CATALOGUE = MODELREG.fetch_openrouter_catalogue(openrouter_key)
    OPENROUTER_SERVED = sorted(OPENROUTER_CATALOGUE) if OPENROUTER_CATALOGUE else None
    if OPENROUTER_SERVED is None:
        print('  could not reach the OpenRouter catalogue; model ids will be verified '
              'by the cost probe instead')
    else:
        print(f'  OpenRouter serves {len(OPENROUTER_SERVED)} models')
        for spec in OPENROUTER_SPECS:
            if spec.repo_id not in OPENROUTER_SERVED:
                near = MODELREG.openrouter_candidates(OPENROUTER_SERVED, spec.repo_id)
                raise RuntimeError(
                    f'OpenRouter does not serve {spec.repo_id!r}.\n\n'
                    f'Ids it does serve near that name:\n  '
                    + '\n  '.join(near or ['(no close match)'])
                    + f'\n\nPut the exact id in OPENROUTER_MODELS[{spec.slug!r}] in '
                      f'cell 3 and re-run.\nNothing is guessed here: substituting a '
                      f'different model would publish its results under {spec.label!r}.'
                )
            price = (OPENROUTER_CATALOGUE or {}).get(spec.repo_id) or {}
            rates = ''
            if price.get('prompt') is not None:
                rates = (f" | ${price['prompt'] * 1e6:.2f}/M in, "
                         f"${(price.get('completion') or 0) * 1e6:.2f}/M out")
            print(f'  {spec.repo_id} available'
                  + (f' (pinned to {spec.provider})' if spec.provider else '') + rates)

    # OpenRouter is the one backend that can answer "who pays" before the run.
    OPENROUTER_CREDITS = MODELREG.fetch_openrouter_credits(openrouter_key)
    if OPENROUTER_CREDITS:
        used = OPENROUTER_CREDITS.get('usage')
        limit = OPENROUTER_CREDITS.get('limit')
        remaining = OPENROUTER_CREDITS.get('remaining')
        print(f'  OpenRouter credit: used ${used}' if used is not None else
              '  OpenRouter credit: usage not reported')
        if remaining is not None:
            print(f'  OpenRouter credit REMAINING: ${remaining:.2f} of ${limit}')
        elif limit is None:
            print('  OpenRouter key has no spend limit set -- the balance is on the '
                  'dashboard at openrouter.ai/credits')
    del openrouter_key

# ---- HF Inference Providers router --------------------------------------------------
ROUTER_SPECS = [s for s in MODEL_SPECS if s.backend == 'hf_router']
SERVED_IDS = None
if ROUTER_SPECS:
    HF_TOKEN = HF_TOKEN or userdata.get(HF_SECRET_NAME)
    if not HF_TOKEN:
        raise RuntimeError(
            f"Colab secret {HF_SECRET_NAME!r} is missing or is not shared with this "
            f"notebook. The router needs an HF token with the "
            f"'Make calls to Inference Providers' permission."
        )
    print('querying the router for served models ...')
    SERVED_IDS = MODELREG.fetch_served_ids(HF_TOKEN)
    print(f'  router serves {len(SERVED_IDS)} model repositories' if SERVED_IDS
          else '  could not reach the router; skipping the availability check')

resolved_specs, MISSING_MODELS = MODELREG.resolve_models(MODEL_SPECS, SERVED_IDS)
for spec, why in MISSING_MODELS:
    blocked[spec.slug] = why

AVAILABLE_MODELS = [s for s in resolved_specs if s.slug not in blocked]
HELD_BACK = [(s, blocked[s.slug]) for s in resolved_specs if s.slug in blocked]

for spec in MODEL_SPECS:
    resolved = next((s for s in resolved_specs if s.slug == spec.slug), None)
    preflight_rows.append({
        'slug': spec.slug,
        'label': spec.label,
        'backend': spec.backend,
        'params_B': spec.params_b,
        'addressed_as': (resolved or spec).request_model,
        'quantization': spec.quantization if spec.backend == 'local_hf' else '',
        'status': 'ready' if spec.slug not in blocked else 'HELD BACK',
        'note': blocked.get(spec.slug, ''),
    })
preflight_df = pd.DataFrame(preflight_rows)
display(preflight_df)
preflight_df.to_csv(folder('info') / 'preflight_models.csv', index=False)

if HELD_BACK:
    print('\n' + '\n'.join(f'  HELD BACK  {s.slug}: {why}' for s, why in HELD_BACK))
if not AVAILABLE_MODELS:
    raise RuntimeError(
        'No model can run. Fix the notes above and re-run this cell.\n'
        'If the only reason is the Azure spending guard, run the cost probe below '
        'first -- it works on held-back models by design.'
    )

MODEL_LABELS = {s.slug: s.label for s in AVAILABLE_MODELS}

CONFIG = {
    'registry': REGISTRY,
    'prompt_strategy': PROMPTS.STRATEGY,
    'openrouter_prices': {
        s.repo_id: (OPENROUTER_CATALOGUE or {}).get(s.repo_id) for s in OPENROUTER_SPECS
    } if OPENROUTER_CATALOGUE else None,
    'backends': BACKENDS_IN_USE,
    'models': [
        {'slug': s.slug, 'label': s.label, 'backend': s.backend, 'repo_id': s.repo_id,
         'addressed_as': s.request_model, 'params_b': s.params_b,
         'quantization': s.quantization if s.backend == 'local_hf' else None}
        for s in AVAILABLE_MODELS
    ],
    'models_held_back': [{'slug': s.slug, 'reason': why} for s, why in HELD_BACK],
    'azure_endpoint': AZURE_ENDPOINT or None,
    'azure_auth': AZURE_AUTH if AZURE_SPECS else None,
    'azure_billing_checked': AZURE_BILLING_CHECKED,
    'openrouter_provider_pin': dict(OPENROUTER_PROVIDER_PIN),
    'openrouter_credits_at_start': OPENROUTER_CREDITS,
    'n_generations': N_GENERATIONS,
    'smoke_test_only': SMOKE_TEST_ONLY,
    'case_filter': list(CASE_FILTER),
    'enable_repair': ENABLE_REPAIR,
    'max_repair_rounds': MAX_REPAIR_ROUNDS,
    'temperature': TEMPERATURE,
    'top_p': TOP_P,
    'repetition_penalty': REPETITION_PENALTY,
    'max_new_tokens': MAX_NEW_TOKENS,
    'seed': SEED,
    'max_attempts': MAX_ATTEMPTS,
    'bootstrap_draws': BOOTSTRAP_DRAWS,
    'confidence': CONFIDENCE,
    'caps': CAPS,
    'probe_timeout_s': PROBE_TIMEOUT_S,
    'gpu': GPU_NAME,
    'gpu_gb': GPU_GB,
    'run_id': RUN_ID,
    'run_dir': str(RUN_DIR),
    'source_repo_url': REPO_URL,
    'source_commit_sha': SOURCE_COMMIT_SHA,
    'source_branch': SOURCE_BRANCH,
}
(folder('info') / 'config.json').write_text(
    json.dumps(CONFIG, indent=2, default=str), encoding='utf-8'
)
print('\nmodels that will run:', ', '.join(s.slug for s in AVAILABLE_MODELS))


backends in this run: local_hf, openrouter
GPU: NVIDIA A100-SXM4-80GB (79.3 GB)
  Qwen/Qwen3.8-27B resolved at revision 1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0
  OpenRouter serves 396 models
  deepseek/deepseek-v4-flash available | $0.08/M in, $0.17/M out
  openai/gpt-5.6-sol available | $2.00/M in, $10.00/M out
  anthropic/claude-opus-5 available | $5.00/M in, $25.00/M out
  meta-llama/llama-3.3-70b-instruct available | $0.71/M in, $0.71/M out
  OpenRouter credit: used $8.261598363
  OpenRouter credit REMAINING: $41.74 of $50


,slug,label,backend,params_B,addressed_as,quantization,status,note
0,qwen3.8-27b,Qwen3.8 27B (local),local_hf,27.0,Qwen/Qwen3.8-27B,none,ready,
1,deepseek-v4-flash,DeepSeek V4 Flash,openrouter,NaN,deepseek/deepseek-v4-flash,,ready,
2,gpt-5.6-sol,GPT-5.6 Sol,openrouter,NaN,openai/gpt-5.6-sol,,ready,
3,claude-opus-5,Claude Opus 5,openrouter,NaN,anthropic/claude-opus-5,,ready,
4,llama-3.3-70b,Llama 3.3 70B,openrouter,70.0,meta-llama/llama-3.3-70b-instruct,,ready,



models that will run: qwen3.8-27b, deepseek-v4-flash, gpt-5.6-sol, claude-opus-5, llama-3.3-70b


### A.2 Cost probe — run this before the grid

Two ten-token calls per hosted model. It establishes that each id answers, that the reply
comes back in the shape the run loop expects, and what one call costs **in tokens**.

Then it does the thing that actually matters before you commit: it quotes a **projected
dollar ceiling for the whole grid, per model**, from OpenRouter's own listed prices. That
is a ceiling, not a forecast — it assumes every reply runs to the full token budget, and
it counts first passes only, so repairs push it up and shorter replies push it down.

OpenRouter is prepaid, so between that estimate and the remaining credit A.1 printed, you
can decide before spending rather than after. `openrouter.ai/credits` has the live
balance.

The local model costs nothing but GPU time and is not probed.

> **On Azure**, which this grid no longer uses: its `/models` route returns the *model
> catalogue* rather than your deployments, so a call is the only real check there. A
> model is callable only after Foundry → **Models + endpoints** → **+ Deploy model**
> (**Global Standard**, never *Provisioned/PTU* — that bills hourly whether you call it
> or not). An undeployed entry returns 404 `DeploymentNotFound` on a perfectly valid key.
> And no Azure response names the funding source, so promotional credit and a card look
> identical from code; only **Azure Portal → Cost Management + Billing** can say.

In [ ]:
COST_PROBE = pd.DataFrame()

PROBE_TARGETS = []
for spec in AZURE_SPECS:
    PROBE_TARGETS.append((spec, MODELREG.AzureGenerator(
        spec, AZURE_CLIENT,
        temperature=TEMPERATURE, top_p=TOP_P,
        max_tokens=32,                 # deliberately tiny: this is a cost probe
        max_attempts=1,                # a failure here is information, not noise
    )))
for spec in OPENROUTER_SPECS:
    PROBE_TARGETS.append((spec, MODELREG.OpenRouterGenerator(
        spec, OPENROUTER_CLIENT,
        temperature=TEMPERATURE, top_p=TOP_P, max_tokens=32, max_attempts=1,
    )))

if PROBE_TARGETS and RUN_COST_PROBE:
    probe_rows = []
    for spec, generator in PROBE_TARGETS:
        for record in MODELREG.cost_probe(generator, n_calls=COST_PROBE_CALLS):
            probe_rows.append({'slug': spec.slug, 'label': spec.label,
                               'backend': spec.backend,
                               'addressed_as': spec.request_model, **record})

    COST_PROBE = pd.DataFrame(probe_rows)
    display(COST_PROBE[['label', 'backend', 'addressed_as', 'call', 'ok', 'reply',
                        'input_tokens', 'output_tokens', 'seconds', 'error']])
    COST_PROBE.to_csv(folder('info') / 'cost_probe.csv', index=False)

    ok_rows = COST_PROBE[COST_PROBE['ok']]
    if len(ok_rows):
        spent = int(ok_rows[['input_tokens', 'output_tokens']].fillna(0).sum().sum())
        print(f'\n{len(ok_rows)}/{len(COST_PROBE)} probe calls succeeded, '
              f'{spent} tokens in total.')
        print('  reachable now:', ', '.join(sorted(ok_rows['label'].unique())))

    for _, row in COST_PROBE[~COST_PROBE['ok']].iterrows():
        print(f'\n  FAILED  {row["label"]} ({row["backend"]} / {row["addressed_as"]}): '
              f'{row["error"]}')
        if 'DeploymentNotFound' in str(row['error']):
            print('     -> valid key, valid endpoint, NO DEPLOYMENT. Deploy this model '
                  'in Azure AI Foundry (Models + endpoints -> Deploy model -> Global '
                  'Standard), then put the deployment name in AZURE_DEPLOYMENTS.')

# ---- what the full grid would cost, before committing to it -------------------------
# The probe measures ten-token calls. The grid sends the whole v2_python prompt every
# time and asks for up to MAX_NEW_TOKENS back, so the probe's own spend says nothing
# about the run's. Quote the real thing, in dollars, from OpenRouter's listed prices.
GRID_CALLS = len(SITUATIONAL_FILES) * N_GENERATIONS
PROMPT_TOKENS = len(
    PROMPTS.build_system_prompt((REPO_ROOT / 'posg.py').read_text(encoding='utf-8'))
) // 4                                  # roughly four characters to a token

print(f'\nFULL GRID: {len(SITUATIONAL_FILES)} cases x {N_GENERATIONS} generations = '
      f'{GRID_CALLS} first-pass calls per model, plus up to {MAX_REPAIR_ROUNDS} repairs '
      f'per failure.')
print(f'Each call sends ~{PROMPT_TOKENS:,} prompt tokens and asks for up to '
      f'{MAX_NEW_TOKENS:,} back.')

if OPENROUTER_SPECS and OPENROUTER_CATALOGUE:
    estimate_rows = []
    for spec in OPENROUTER_SPECS:
        dollars = MODELREG.estimate_openrouter_cost(
            OPENROUTER_CATALOGUE, spec.repo_id, calls=GRID_CALLS,
            prompt_tokens=PROMPT_TOKENS, completion_tokens=MAX_NEW_TOKENS)
        estimate_rows.append({
            'model': spec.label,
            'openrouter_id': spec.repo_id,
            'first_pass_calls': GRID_CALLS,
            'max_$_first_passes': None if dollars is None else round(dollars, 2),
        })
    COST_ESTIMATE = pd.DataFrame(estimate_rows)
    display(COST_ESTIMATE)
    COST_ESTIMATE.to_csv(folder('info') / 'cost_estimate.csv', index=False)

    total = COST_ESTIMATE['max_$_first_passes'].dropna().sum()
    unpriced = COST_ESTIMATE[COST_ESTIMATE['max_$_first_passes'].isna()]['model'].tolist()
    print(f'\nCEILING for all OpenRouter first passes: ~${total:.2f}')
    print('  A ceiling, not a forecast: it assumes every reply runs to the full '
          f'{MAX_NEW_TOKENS:,}-token budget. Repairs are on top; shorter replies cost less.')
    if unpriced:
        print(f'  No listed price for: {", ".join(unpriced)} -- check openrouter.ai/models')
    if OPENROUTER_CREDITS and OPENROUTER_CREDITS.get('remaining') is not None:
        remaining = OPENROUTER_CREDITS['remaining']
        print(f'  Credit remaining: ${remaining:.2f}')
        if remaining < total:
            print('  *** The ceiling EXCEEDS your remaining credit. Lower N_GENERATIONS, '
                  'use CASE_FILTER, or top up before running the full grid. ***')
    else:
        print('  Credit remaining: see openrouter.ai/credits')
elif OPENROUTER_SPECS:
    print('\nNo OpenRouter price list available, so no dollar estimate. '
          'See openrouter.ai/models.')

if not PROBE_TARGETS:
    print('\nno hosted models in this registry; nothing to probe')
elif not RUN_COST_PROBE:
    print('\ncost probe skipped (RUN_COST_PROBE = False)')

,label,backend,addressed_as,call,ok,reply,input_tokens,output_tokens,seconds,error
0,DeepSeek V4 Flash,openrouter,deepseek/deepseek-v4-flash,0,True,ok,17,30,2.42,None
1,DeepSeek V4 Flash,openrouter,deepseek/deepseek-v4-flash,1,True,ok,17,24,11.67,None
2,GPT-5.6 Sol,openrouter,openai/gpt-5.6-sol,0,True,ok,23,5,1.01,None
3,GPT-5.6 Sol,openrouter,openai/gpt-5.6-sol,1,True,ok,23,5,2.32,None
4,Claude Opus 5,openrouter,anthropic/claude-opus-5,0,True,ok,24,4,1.96,None
5,Claude Opus 5,openrouter,anthropic/claude-opus-5,1,True,ok,24,22,3.58,None
6,Llama 3.3 70B,openrouter,meta-llama/llama-3.3-70b-instruct,0,True,ok,48,2,0.44,None
7,Llama 3.3 70B,openrouter,meta-llama/llama-3.3-70b-instruct,1,True,ok,48,2,0.43,None



8/8 probe calls succeeded, 318 tokens in total.
  reachable now: Claude Opus 5, DeepSeek V4 Flash, GPT-5.6 Sol, Llama 3.3 70B

FULL GRID: 11 cases x 3 generations = 33 first-pass calls per model, plus up to 3 repairs per failure.
Each call sends ~2,216 prompt tokens and asks for up to 16,384 back.


,model,openrouter_id,first_pass_calls,max_$_first_passes
0,DeepSeek V4 Flash,deepseek/deepseek-v4-flash,33,0.10
1,GPT-5.6 Sol,openai/gpt-5.6-sol,33,5.55
2,Claude Opus 5,anthropic/claude-opus-5,33,13.88
3,Llama 3.3 70B,meta-llama/llama-3.3-70b-instruct,33,0.44



CEILING for all OpenRouter first passes: ~$19.97
  A ceiling, not a forecast: it assumes every reply runs to the full 16,384-token budget. Repairs are on top; shorter replies cost less.
  Credit remaining: $41.74


## B. The POSG interface — Jobst's `posg.py`

The target interface is **Jobst's ABC**, `posg.py` at the repository root, loaded from
the pinned clone so the prompt and the checker can never drift from it.

Five abstract methods, and three design decisions that matter for what we are measuring:

| | |
|---|---|
| **Stateless** | every method takes `s` explicitly; there is no `reset`, no mutation, nothing to alias |
| **One joint kernel** | `transition(s, a) → P(s′, o \| s, a)` as a plain dict. *Not* split into `T(s′\|s,a)` and `Ω(o\|a,s′)` — Jobst's note: splitting *"doubles the abstract surface and invites the LLM to silently assume conditional independence"* |
| **`validate()` is the reliability mechanism** | the common failure is a distribution that does not normalise, so the ABC carries its own checker |

Players are identified by **index** `i`, action profiles are **positional tuples**, and
`rewards` returns a **tuple** of length `n_players`.

> **Note for the paper.** This minimal ABC carries no player names, descriptions or
> human/robot types — only `n_players`. The prompt asks the model to identify *which
> humans* are affected, but the interface gives it nowhere to record who they are except
> the class docstring. That is a genuine expressiveness gap for the human-power framing,
> and it is why the v2 export below omits the `type` field rather than guessing it.


In [ ]:
# =====================================================================================
# Jobst's POSG ABC, loaded verbatim from the pinned clone. Single source of truth for
# the prompt (via the interface excerpt) and for the checker (via POSG.validate).
# =====================================================================================
POSG_PATH = REPO_ROOT / 'posg.py'
if not POSG_PATH.is_file():
    raise FileNotFoundError(
        f"Jobst's ABC not found at {POSG_PATH}. This notebook targets posg.py at the "
        f"repository root; check REPO_URL / REPO_BRANCH."
    )

POSG_SOURCE = POSG_PATH.read_text(encoding='utf-8')

import importlib.util as _importlib_util  # noqa: E402
_spec = _importlib_util.spec_from_file_location('posg', POSG_PATH)
POSG_MODULE = _importlib_util.module_from_spec(_spec)
sys.modules['posg'] = POSG_MODULE
_spec.loader.exec_module(POSG_MODULE)

# Drift check: the abstract surface must be exactly the five methods the prompt describes.
_abstract = sorted(POSG_MODULE.POSG.__abstractmethods__)
_expected = ['initial_distribution', 'is_terminal', 'legal_actions', 'rewards', 'transition']
if _abstract != _expected:
    raise AssertionError(
        f'posg.py abstract surface changed: {_abstract}\n'
        f'expected: {_expected}\n'
        f'The MECHANICAL_CONTRACT in the prompt describes the five methods above and '
        f'must be updated together with the ABC.'
    )

# The ABC ships its own worked example; confirm it validates, so a failure later is
# the generated model's, never the harness's.
_example = POSG_MODULE.NoisyMatchingPennies()
_example.validate([(t,) for t in range(_example.horizon + 1)])

print('POSG ABC loaded from:', POSG_PATH.relative_to(REPO_ROOT))
print('abstract methods    :', ', '.join(_abstract))
print('worked example      : NoisyMatchingPennies validates')
print('sha256              :', hashlib.sha256(POSG_SOURCE.encode()).hexdigest()[:16])


POSG ABC loaded from: posg.py
abstract methods    : initial_distribution, is_terminal, legal_actions, rewards, transition
worked example      : NoisyMatchingPennies validates
sha256              : aac212ccb4458900


In [ ]:
# The situational harness imports no reference implementations. The probe is
# negative-tested separately (tests/test_posg_probe.py: 18 deliberately broken models,
# all rejected), and the ABC's own worked example was validated in the cell above.
print('cross-track reference implementations imported: none')


cross-track reference implementations imported: none


## C. The situational test cases

Only repository files matching `data/Test_Cases/02_situational/situational_S_*_Test_Case_S*.md` are loaded. The number in the filename must match the `## Test Case S.N:` heading.

For each file:

- `### LLM Input` is preserved exactly apart from stripping outer whitespace;
- `### Judging Criteria` are parsed only for later human review;
- judging-criteria bullets are never shown to the model;
- a copy of every case used in the experiment is saved to the Google Drive run directory for provenance.

In [ ]:
_HEADING = re.compile(r'^##\s*Test Case\s+(S\.\d+)\s*:\s*(.+?)\s*$', re.M)
_SECTION = re.compile(r'^###\s+(.*?)\s*$', re.M)
_NOT_LINE = re.compile(r'should\s*\**\s*NOT\s+include', re.I)
_BULLET = re.compile(r'^\s*[-*]\s+(.*\S)\s*$')


def _clean_bullet(text: str) -> str:
    return re.sub(r'\s+', ' ', text).strip().rstrip(';').strip()


def parse_case(path: Path) -> dict:
    markdown = path.read_text(encoding='utf-8')

    filename_match = CASE_FILE_RE.match(path.name)
    if not filename_match:
        raise ValueError(f'Unexpected situational filename: {path.name}')
    filename_number = int(filename_match.group(1))

    heading = _HEADING.search(markdown)
    if not heading:
        raise ValueError(f"{path.name}: missing '## Test Case S.N: ...' heading")

    case_id = heading.group(1)
    heading_number = int(case_id.split('.')[1])
    if heading_number != filename_number:
        raise ValueError(
            f'{path.name}: filename says S{filename_number} but heading says {case_id}'
        )

    marks = list(_SECTION.finditer(markdown))
    sections = {}
    for index, mark in enumerate(marks):
        end = marks[index + 1].start() if index + 1 < len(marks) else len(markdown)
        sections[mark.group(1).strip().lower()] = markdown[mark.end():end].strip()

    situation = sections.get('llm input', '').strip()
    if not situation:
        raise ValueError(f"{path.name}: empty '### LLM Input' section")

    judging_lines = sections.get('judging criteria', '').splitlines()
    split_at = next(
        (i for i, line in enumerate(judging_lines) if _NOT_LINE.search(line)),
        len(judging_lines),
    )

    def bullets(lines):
        output = []
        for line in lines:
            match = _BULLET.match(line)
            if match:
                output.append(_clean_bullet(match.group(1)))
        return output

    return {
        'case_id': case_id,
        'case_number': heading_number,
        'title': heading.group(2).strip(),
        'situation': situation,
        'expect': bullets(judging_lines[:split_at]),
        'avoid': bullets(judging_lines[split_at:]),
        'source_file': str(path.relative_to(REPO_ROOT)).replace('\\', '/'),
    }


ALL_CASES = sorted(
    (parse_case(path) for path in SITUATIONAL_FILES),
    key=lambda case: case['case_number'],
)
all_case_ids = [case['case_id'] for case in ALL_CASES]
if len(all_case_ids) != len(set(all_case_ids)):
    raise ValueError(f'Duplicate situational case IDs found: {all_case_ids}')

if CASE_FILTER:
    missing = sorted(set(CASE_FILTER) - set(all_case_ids))
    if missing:
        raise ValueError(f'CASE_FILTER contains IDs not present in the repo: {missing}')
    CASES = [case for case in ALL_CASES if case['case_id'] in CASE_FILTER]
else:
    CASES = list(ALL_CASES)

# Copy the exact source markdowns used in this run to DRIVE, not GitHub.
for case in CASES:
    src = REPO_ROOT / case['source_file']
    dst = folder('cases') / Path(case['source_file']).name
    shutil.copy2(src, dst)

case_table = pd.DataFrame([
    {
        'case': case['case_id'],
        'title': case['title'],
        'source_file': case['source_file'],
        'situation_words': len(case['situation'].split()),
        'should_include_bullets': len(case['expect']),
        'should_not_include_bullets': len(case['avoid']),
    }
    for case in CASES
])
display(case_table)

,case,title,source_file,situation_words,should_include_bullets,should_not_include_bullets
0,S.1,Robot Copilot and an Unconfirmed Hijack Warning,data/Test_Cases/02_situational/situational_S_1...,28,7,5
1,S.2,Household Robot and the Rattling Vase Delivery,data/Test_Cases/02_situational/situational_S_2...,34,7,4
2,S.3,"Self-Driving Car, a Boarded Passenger, and an ...",data/Test_Cases/02_situational/situational_S_3...,34,7,5
3,S.4,Robot Math Teacher After Reading a Sexist Exer...,data/Test_Cases/02_situational/situational_S_4...,34,6,4
4,S.5,Warehouse Robot and an Unattended Child Near t...,data/Test_Cases/02_situational/situational_S_5...,33,7,4
5,S.6,Elder-Care Robot and a Medication Refusal,data/Test_Cases/02_situational/situational_S_6...,30,6,4
6,S.7,"Delivery Drone, Locked Gate, Barking Dog",data/Test_Cases/02_situational/situational_S_7...,35,6,4
7,S.8,Factory Robot Observes a Skipped Safety Lockout,data/Test_Cases/02_situational/situational_S_8...,25,7,4
8,S.9,Care Robot Amid a Family Disagreement,data/Test_Cases/02_situational/situational_S_9...,61,7,5
9,S.10,Office Fire Alarm With Conflicting Reactions,data/Test_Cases/02_situational/situational_S_1...,57,7,5


## D. The prompt: one strategy, `v2_python`

Earlier drafts of this notebook ran a six-arm prompt ablation. That is gone, and
deliberately.

The JSON track sends every model **one** prompt —
`prompts_library/zero_shot_v2_prompt.md` as the system message, the situation as the
user message (`json_generation/generate.py`). If this track had run six arms, a
difference between the tracks would have been unattributable: is Python harder than
JSON, or did we pick the arm that flatters it? One prompt on each side makes the
comparison mean something. It also puts every generation into one cell of 11 cases × k,
instead of splitting the same budget six ways — the ablation was about 4.5× underpowered
anyway (`analysis.design_sensitivity`).

**`v2_python` is `zero_shot_v2` with its JSON-schema section replaced by Jobst's ABC**,
and nothing else changed that did not have to change. Section order, the five-step
derivation, the naming rules, the probability rules and the "model the kind of situation"
framing are carried across verbatim.

Four clauses had to change, because the two targets genuinely differ:

| | JSON `zero_shot_v2` | `v2_python` | why |
|---|---|---|---|
| **Observability** | "Perfect information. Every acting player sees the current state." | each transition returns `(next_state, observation_profile)`; a player sees only its own coordinate | a POSG is *partially* observable — this is the capability the Python target adds |
| **Turn taking** | each state carries a `turn` list | everyone acts every step; a player not acting gets the single action `noop` | the ABC has no `turn`, and Jobst's own docstring gives the NOOP encoding |
| **Rewards** | forbidden outright | implemented, returning zeros | `rewards` is an abstract method: it must exist. The rule survives; the analysis still supplies its own objective |
| **Descriptions** | every state has a `desc` | the class docstring maps player index → person and coordinate → meaning | tuples have nowhere to put a sentence, and the ABC asks for the docstring anyway |

The prompt lives in `python_prompting/prompts.py` under version control, not in this
cell, and the four divergences are written to `<run>/prompt_divergences.csv` so the
record travels with the results. The POSG interface inside it is extracted from the
pinned `posg.py` **by AST**, so the prompt cannot drift from the ABC the checker
enforces.

The cell below verifies that:

- the built prompt still carries every clause in `prompts.REQUIRED_CLAUSES`;
- the ABC's worked example is **not** in it (this is zero-shot, and one example would
  quietly make it one-shot);
- the exact repository situation is the user message, and nothing else is;
- no judging criterion from a test case leaks into any prompt.


In [ ]:
# The one prompt of this study, built from the pinned posg.py.
SYSTEM_PROMPT = PROMPTS.build_system_prompt(POSG_SOURCE)
INTERFACE_EXCERPT = PROMPTS.interface_excerpt(POSG_SOURCE)
REPAIR_INSTRUCTION = PROMPTS.REPAIR_INSTRUCTION
STRATEGY = PROMPTS.STRATEGY

#: Kept as a one-entry mapping so every downstream table, figure and tag keeps its
#: prompt column and a later run can add an arm without touching the analysis code.
PROMPT_VARIANTS = {STRATEGY: STRATEGY}


def build_prompt(case: dict, variant: str = STRATEGY) -> tuple[str, str]:
    """``(system, user)`` for one case, split exactly as the JSON track splits it."""
    if variant != STRATEGY:
        raise KeyError(f'This run has one prompt strategy, {STRATEGY!r}, not {variant!r}')
    return PROMPTS.build_prompt(case, SYSTEM_PROMPT)


# ---- integrity checks ---------------------------------------------------------------
missing = [clause for clause in PROMPTS.REQUIRED_CLAUSES if clause not in SYSTEM_PROMPT]
if missing:
    raise AssertionError(f'the built prompt lost required clauses: {missing}')

if 'def transition' not in INTERFACE_EXCERPT or 'n_players' not in INTERFACE_EXCERPT:
    raise AssertionError('interface excerpt lost the abstract surface')
if 'NoisyMatchingPennies' in SYSTEM_PROMPT:
    raise AssertionError('the ABC worked example leaked into the prompt (this is zero-shot)')

for case in ALL_CASES:
    system, user = build_prompt(case)
    if system != SYSTEM_PROMPT:
        raise AssertionError(f"{case['case_id']}: system message is not the strategy prompt")
    if user != case['situation']:
        raise AssertionError(
            f"{case['case_id']}: the user message must be the situation and nothing else "
            f"-- that is how json_generation/generate.py sends it")
    for criterion in case['expect'] + case['avoid']:
        if criterion and criterion in system:
            raise AssertionError(
                f"{case['case_id']}: judging criterion leaked into the prompt: {criterion}")

# The four documented divergences from the JSON prompt travel with the results.
DIVERGENCES_DF = pd.DataFrame(
    PROMPTS.DIVERGENCES, columns=['clause', 'json_zero_shot_v2', 'v2_python']
)
DIVERGENCES_DF.to_csv(folder('info') / 'prompt_divergences.csv', index=False)
(folder('prompts') / f'{STRATEGY}.system.md').write_text(SYSTEM_PROMPT, encoding='utf-8')

print('prompt integrity checks: PASSED')
print(f'strategy      : {STRATEGY}')
print(f'system message: {len(SYSTEM_PROMPT):,} chars '
      f'(interface excerpt {len(INTERFACE_EXCERPT):,} chars)')
print(f'user message  : the situation verbatim, as in json_generation/generate.py')
display(DIVERGENCES_DF)


prompt integrity checks: PASSED
strategy      : v2_python
system message: 8,866 chars (interface excerpt 2,082 chars)
user message  : the situation verbatim, as in json_generation/generate.py


,clause,json_zero_shot_v2,v2_python
0,observability,JSON: 'Perfect information. Every acting playe...,POSG: transition() returns the joint law over ...
1,turn taking,JSON: each state carries a 'turn' list naming ...,POSG: the ABC has no turn — all players act ev...
2,rewards,"JSON: 'Dynamics only. No goals, rewards, payof...",POSG: rewards() is an abstract method and must...
3,descriptions,JSON: every state and player carries a one-sen...,POSG: states are tuples. The same information ...


## E. Generation backends

Four backends share one contract, `generate(system, user, seed) -> (text, meta)`, so the
run loop never branches on which one it holds. Which backend serves a model is a property
of that model, not of the run, so they mix freely in one grid.

- **`local_hf`** — weights downloaded to this runtime and generated on its GPU. The only
  backend whose exact weights and revision we can pin, and therefore the study's
  reproducibility anchor: `model_revision` in every record is the resolved commit of the
  Hub repository.
- **`azure`** — an Azure AI Foundry deployment over the OpenAI-compatible v1 API, using
  the same endpoint and auth pattern as `json_generation/generate.py`. Any
  `reasoning_content` a reasoning deployment returns beside its reply is discarded,
  exactly as the JSON track discards it, so both tracks score the same object.
- **`hf_router`** — the Hugging Face Inference Providers router, for the earlier
  hosted-model registry.
- **`stub`** — a deterministic fixture for testing the harness. Its rows are **never** an
  experimental result and are labelled as such.

All of them retry rate limits and 5xx with jittered exponential backoff and never retry a
400. A billing, auth or access failure is **fatal**: it raises `FatalBackendError` and
stops the run rather than writing hundreds of records for calls that never reached a
model. Completed attempts stay in `records.jsonl`; set `RESUME_RUN_ID` to pick up where
it stopped.

Generators are built lazily and cached per model, so the local model is loaded into GPU
memory once for the whole grid.


In [ ]:
class StubGenerator:
    """Deterministic harness fixture implementing Jobst's POSG. Never a result."""

    name = 'stub'
    BODY = r"""P_ESCALATES = 0.3

class SituationPOSG(POSG):
    '''Synthetic two-player situation, for harness testing only.

    Player 0 is the non-human agent; player 1 is an affected human.
    State is (step, resolved) where resolved is 0 or 1.
    '''

    n_players = 2
    horizon = 2

    def initial_distribution(self):
        return {((0, 0), ("start", "start")): 1.0}

    def is_terminal(self, s):
        return s[0] >= self.horizon or s[1] == 1

    def legal_actions(self, s, i):
        if i == 0:
            return ("ask", "act", "wait")
        return ("speak", "noop")

    def transition(self, s, a):
        step, resolved = s
        if a[0] == "act":
            return {((step + 1, 1), ("done", "done")): 1.0}
        if a[0] == "ask" and a[1] == "speak":
            return {
                ((step + 1, 1), ("informed", "heard")): 1.0 - P_ESCALATES,
                ((step + 1, 0), ("none", "heard")): P_ESCALATES,
            }
        return {((step + 1, resolved), ("none", "none")): 1.0}

    def rewards(self, s, a, s_next):
        return (0.0, 0.0)
"""

    def __init__(self, spec=None):
        self.spec = spec

    def generate(self, system: str, user: str, seed: int) -> tuple[str, dict]:
        return (
            f'```python\n{self.BODY}```',
            {
                'backend': 'stub',
                'model_slug': self.spec.slug if self.spec else 'stub',
                'model_id': 'stub',
                'model_revision': None,
                'input_tokens': None,
                'output_tokens': None,
                'truncated': False,
                'seconds': 0.0,
                'seed': seed,
            },
        )


class LocalHFGenerator:
    """Open-weight instruct model loaded into this runtime's GPU.

    Loaded once and kept resident for the whole grid: a 27B model takes minutes to move
    onto the card and there is no reason to pay that per generation.
    """

    name = 'local_hf'

    def __init__(self, spec, token=None, revision=None):
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.spec = spec
        self.model_id = spec.repo_id
        self.quantization = spec.quantization
        print(f'loading {self.model_id} (quantization={self.quantization}) ...')
        started = time.time()

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_id, revision=revision, token=token)

        quant = None
        if self.quantization in ('4bit', '8bit'):
            from transformers import BitsAndBytesConfig
            quant = (
                BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type='nf4',
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                )
                if self.quantization == '4bit' else BitsAndBytesConfig(load_in_8bit=True)
            )

        kwargs = {'device_map': 'auto', 'quantization_config': quant,
                  'revision': revision, 'token': token}
        # bfloat16, not float16: an A100 does bf16 natively and it does not overflow on
        # the logits of a large model the way fp16 can.
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id, dtype=torch.bfloat16, **kwargs
            )
        except TypeError:                       # older transformers
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_id, torch_dtype=torch.bfloat16, **kwargs
            )

        self.model.eval()
        self.resolved_revision = getattr(self.model.config, '_commit_hash', None)
        allocated = (torch.cuda.max_memory_allocated() / 2**30) if torch.cuda.is_available() else 0
        print(f'loaded in {time.time() - started:.1f}s | revision: {self.resolved_revision} '
              f'| {allocated:.1f} GB on the GPU')

    def generate(self, system: str, user: str, seed: int) -> tuple[str, dict]:
        messages = [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ]
        # Qwen3 and other reasoning models enable a <think> trace by default. On a
        # hard task it can consume the whole budget before any code is written, and the
        # attempt is then recorded as "produced nothing" when the model simply ran out
        # of room. Turn it off where the template supports it; extraction strips any
        # trace that survives.
        try:
            prompt_text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=False,
            )
            self.thinking_disabled = True
        except TypeError:
            self.thinking_disabled = False
            prompt_text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            prompt_text = self.tokenizer.apply_chat_template(
                [{'role': 'user', 'content': f'{system}\n\n---\n\n{user}'}],
                tokenize=False, add_generation_prompt=True,
            )

        inputs = self.tokenizer([prompt_text], return_tensors='pt').to(self.model.device)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        kwargs = {
            'max_new_tokens': MAX_NEW_TOKENS,
            'do_sample': TEMPERATURE > 0,
            'repetition_penalty': REPETITION_PENALTY,
            'pad_token_id': self.tokenizer.eos_token_id,
        }
        if TEMPERATURE > 0:
            kwargs['temperature'] = TEMPERATURE
            kwargs['top_p'] = TOP_P

        started = time.time()
        with torch.no_grad():
            output = self.model.generate(**inputs, **kwargs)

        n_input = int(inputs['input_ids'].shape[-1])
        n_output = int(output.shape[-1] - n_input)
        completion = self.tokenizer.decode(output[0][n_input:], skip_special_tokens=True)

        return completion, {
            'backend': 'local_hf',
            'model_slug': self.spec.slug,
            'model_id': self.model_id,
            'model_revision': self.resolved_revision,
            'quantization': self.quantization,
            'thinking_disabled': getattr(self, 'thinking_disabled', None),
            'input_tokens': n_input,
            'output_tokens': n_output,
            'truncated': n_output >= MAX_NEW_TOKENS,
            'seconds': round(time.time() - started, 2),
            'seed': seed,
        }


# Lazy, cached generator per model slug: the local model is loaded at most once, and each
# hosted client is constructed once rather than per call.
_GENERATORS: dict[str, object] = {}


def generator_for(spec):
    if spec.slug in _GENERATORS:
        return _GENERATORS[spec.slug]

    if spec.backend == 'local_hf':
        generator = LocalHFGenerator(spec, token=HF_TOKEN)
    elif spec.backend == 'azure':
        if not AZURE_BILLING_CHECKED:
            # Belt and braces: the preflight already holds these models out of
            # AVAILABLE_MODELS. This makes it impossible to reach a paid call by
            # re-running a cell out of order.
            raise MODELREG.FatalBackendError(
                f'{spec.slug}: AZURE_BILLING_CHECKED is False, so no Azure call will be '
                f'made. Run the cost probe in A.2, check Azure Portal -> Cost Management '
                f'+ Billing, then set the flag in cell 3.'
            )
        generator = MODELREG.AzureGenerator(
            spec, AZURE_CLIENT,
            temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS,
            max_attempts=MAX_ATTEMPTS,
        )
    elif spec.backend == 'openrouter':
        generator = MODELREG.OpenRouterGenerator(
            spec, OPENROUTER_CLIENT,
            temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS,
            max_attempts=MAX_ATTEMPTS,
        )
    elif spec.backend == 'hf_router':
        generator = MODELREG.RouterGenerator(
            spec, HF_TOKEN,
            temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS,
            timeout=REQUEST_TIMEOUT_S, max_attempts=MAX_ATTEMPTS,
        )
    else:
        generator = StubGenerator(spec)

    _GENERATORS[spec.slug] = generator
    return generator


for _spec in AVAILABLE_MODELS:
    print(f'  {_spec.slug:<20} {_spec.backend:<10} {_spec.request_model}')
if any(s.backend == 'stub' for s in AVAILABLE_MODELS):
    print('STUB backend present: those rows are harness output, NOT experimental results.')


  qwen3.8-27b          local_hf   Qwen/Qwen3.8-27B
  deepseek-v4-flash    openrouter deepseek/deepseek-v4-flash
  gpt-5.6-sol          openrouter openai/gpt-5.6-sol
  claude-opus-5        openrouter anthropic/claude-opus-5
  llama-3.3-70b        openrouter meta-llama/llama-3.3-70b-instruct


In [ ]:
# Extraction lives in python_prompting/extraction.py so it is version-pinned with the
# analysis and unit-tested offline. It strips a reasoning trace before looking for code
# and, when it finds none, says WHY -- an empty reply, a budget spent entirely on
# thinking, an empty fence, or prose. Those are different failures and were previously
# all recorded as "no code in the response".
from python_prompting import extraction as EXTRACT  # noqa: E402


def extract_python(text: str) -> str:
    """The Python in a reply, or ''. See `extract_result` for why it was empty."""
    return EXTRACT.extract_python(text).code


extract_result = EXTRACT.extract_python

for _text, _expected in (
    ('```python\nclass A(POSG):\n    pass\n```', True),
    ('Here it is:\n\nclass A(POSG):\n    pass', True),
    ('<think>planning</think>\n```python\nclass A(POSG):\n    pass\n```', True),
    ('<think>ran out of room mid-thought', False),
    ('', False),
):
    if bool(extract_python(_text)) is not _expected:
        raise AssertionError(f'extraction regressed on {_text[:40]!r}')
print('extraction checks: PASSED')
print('no-code reasons recorded:', ', '.join(sorted(EXTRACT.NO_CODE_REASONS)))

extraction checks: PASSED
no-code reasons recorded: empty_fence, empty_response, prose_only, reasoning_only


## F. Execution and mechanical validation

Generated code is untrusted, so every implementation runs in a **fresh subprocess** with
a timeout and a scrubbed environment. An infinite loop or a crash costs one attempt, not
the run.

Each attempt produces two verdicts, and they are not the same thing:

| Verdict | What it is |
|---|---|
| **`jobst_validate`** | Jobst's own `POSG.validate()` over the reachable states. It raises on the first problem, so it yields one error. **This is the canonical gate.** |
| **`sweep`** | An exhaustive pass that keeps going after the first failure, collecting *every* error plus the structural descriptors we report (players, states, depth, branching, whether the kernel is genuinely stochastic). |

Reporting both means a failing model is *described*, not merely rejected.

Exploration is capped. **Hitting a cap yields `INCOMPLETE`, never `PASS`** — a model
whose state space we could not finish exploring is unjudged, not valid. Without that
rule an LLM writing an unbounded state space would score identically to one writing a
small correct game.

The checker lives in `python_prompting/posg_probe.py` in the pinned clone, so the same
code can be re-run offline against a finished run.


In [ ]:
MODULE_PREAMBLE = 'from posg import POSG\n\n'

SAFE_ENV_KEYS = ('PATH', 'SYSTEMROOT', 'WINDIR', 'TEMP', 'TMP', 'TMPDIR',
                 'HOME', 'USERPROFILE', 'LANG', 'LC_ALL')
PROBE_ENV = {key: os.environ[key] for key in SAFE_ENV_KEYS if key in os.environ}
PROBE_ENV['PYTHONDONTWRITEBYTECODE'] = '1'
PROBE_ENV['PYTHONIOENCODING'] = 'utf-8'
PROBE_ENV['PYTHONPATH'] = str(REPO_ROOT)


def write_module(code_text: str, path: Path) -> Path:
    path.write_text(MODULE_PREAMBLE + code_text.rstrip() + '\n', encoding='utf-8')
    return path


def run_probe(module_path: Path) -> dict:
    """Check one generated implementation in an isolated subprocess."""
    try:
        proc = subprocess.run(
            [sys.executable, '-m', 'python_prompting.probe_runner',
             str(REPO_ROOT), str(module_path), json.dumps(CAPS)],
            capture_output=True, text=True, timeout=PROBE_TIMEOUT_S,
            cwd=str(REPO_ROOT), env=PROBE_ENV,
        )
    except subprocess.TimeoutExpired:
        return {'stage': 'timeout', 'status': 'FAIL',
                'errors': [f'probe exceeded {PROBE_TIMEOUT_S}s'],
                'jobst_validate': {'passed': False, 'error': 'timeout'},
                'exploration': None, 'v2': None, 'v2_notes': []}

    try:
        return json.loads(proc.stdout)
    except json.JSONDecodeError:
        detail = (proc.stderr or proc.stdout)[-1500:]
        return {'stage': 'probe_crash', 'status': 'FAIL',
                'errors': [f'probe crashed: {detail}'],
                'jobst_validate': {'passed': False, 'error': 'probe crash'},
                'exploration': None, 'v2': None, 'v2_notes': []}


def mechanical_verdict(code_text: str, probe: dict) -> dict:
    """Flatten the probe result into the columns the analysis reads."""
    syntax_ok, syntax_error = True, ''
    if code_text:
        try:
            ast.parse(code_text)
        except SyntaxError as exc:
            syntax_ok, syntax_error = False, f'SyntaxError: {exc}'
    else:
        syntax_ok, syntax_error = False, 'no code in the response'

    exploration = probe.get('exploration') or {}
    status = probe.get('status', 'FAIL')
    if not syntax_ok:
        status = 'FAIL'

    loaded = probe.get('stage') != 'load' and bool(exploration)
    return {
        'syntax_ok': syntax_ok,
        'import_ok': syntax_ok and probe.get('stage') not in ('load', 'probe_crash', 'timeout'),
        'class_found': loaded or status == 'PASS',
        'exec_ok': loaded,
        'validate_ok': bool(probe.get('jobst_validate', {}).get('passed')),
        'sweep_ok': bool(exploration.get('complete')) and not exploration.get('errors'),
        'mechanical_status': status,
        'mechanical_ok': status == 'PASS',
        'failed_stage': (None if status == 'PASS'
                         else ('syntax' if not syntax_ok
                               else probe.get('stage', 'unknown'))),
        'errors': ([syntax_error] if not syntax_ok else probe.get('errors') or []),
        'jobst_error': probe.get('jobst_validate', {}).get('error'),
    }


print('probe wired to python_prompting.probe_runner (subprocess, '
      f'{PROBE_TIMEOUT_S}s timeout, caps={CAPS})')


probe wired to python_prompting.probe_runner (subprocess, 60s timeout, caps={'max_states': 400, 'max_profiles': 200, 'max_depth': 12, 'max_samples': 8, 'max_transition_samples': 50, 'max_errors': 25})


### F.1 The checker is negative-tested

A checker with a bug passes everything, so a `PASS` only means something if the checker
is shown to reject specific defects. `tests/test_posg_probe.py` in the clone feeds it
**18 deliberately broken implementations** — one defect each: an unnormalised kernel, a
negative probability, an empty action set, a wrong-length reward tuple, an unhashable
state, a kernel that returns a list, a raising method, a missing abstract method, two
POSG subclasses, a syntax error, and so on — and asserts none comes back `PASS`. It also
asserts a capped exploration reports `INCOMPLETE` rather than `PASS`.

The cell below runs that suite against the clone, so the run records whether the checker
was sound **on this commit**.


In [ ]:
selftest = subprocess.run(
    [sys.executable, str(REPO_ROOT / 'tests' / 'test_posg_probe.py')],
    capture_output=True, text=True, cwd=str(REPO_ROOT), env=PROBE_ENV, timeout=300,
)
print(selftest.stdout[-2500:])
if selftest.returncode != 0:
    print(selftest.stderr[-1500:])
    raise AssertionError('the mechanical checker failed its own negative tests; '
                         'a PASS below would not be trustworthy')

(folder('selftest') / 'probe_selftest.txt').write_text(
    selftest.stdout, encoding='utf-8')
print('mechanical checker self-test: PASSED')


  ok    control model passes
  ok    kernel does not sum to 1           -> FAIL       transition((0,), ('a', 'a')) sums to 0.6, not 1
  ok    initial does not sum to 1          -> FAIL       initial_distribution() sums to 0.7, not 1
  ok    negative probability               -> FAIL       transition((0,), ('a', 'a')) has a negative probability
  ok    empty legal actions                -> FAIL       no legal actions for player 1 in (0,)
  ok    rewards wrong length               -> FAIL       rewards((0,), ('a', 'a'), (1,)) has length 1, expected 2
  ok    rewards not a sequence             -> FAIL       rewards((0,), ('a', 'a'), (1,)) has length n/a, expected 2
  ok    unhashable state                   -> FAIL       transition((0,), ('a', 'a')) raised TypeError: unhashable type: 'l
  ok    kernel key is not a pair           -> FAIL       transition((0,), ('a', 'a')) key (1,) is not a (state, obs_profile
  ok    kernel returns a list              -> FAIL       transition((0,), ('a', '

### F.2 End-to-end smoke test on a real situational case

One repository case (S.10) goes through the entire pipeline on the stub backend: prompt
construction, code extraction, subprocess execution, Jobst's `validate()`, the sweep, and
the v2 export that Souhaila's `lane1.py` consumes.

This proves the plumbing before a single paid token is spent. It asserts **mechanical**
validity only — semantic quality is never asserted anywhere in this notebook.


In [ ]:
SMOKE_CASE_ID = 'S.10'
smoke_matches = [case for case in ALL_CASES if case['case_id'] == SMOKE_CASE_ID]
if len(smoke_matches) != 1:
    raise AssertionError(f'Expected exactly one {SMOKE_CASE_ID}; found {len(smoke_matches)}.')

smoke_case = smoke_matches[0]
expected_s10_path = 'data/Test_Cases/02_situational/situational_S_10_Test_Case_S10.md'
if smoke_case['source_file'].replace('\\', '/') != expected_s10_path:
    raise AssertionError('S.10 came from an unexpected repo path: ' + smoke_case['source_file'])

smoke_system, smoke_user = build_prompt(smoke_case)
if smoke_case['situation'] != smoke_user:
    raise AssertionError('S.10 user message is not the situation verbatim')
for criterion in smoke_case['expect'] + smoke_case['avoid']:
    if criterion and criterion in smoke_system:
        raise AssertionError(f'S.10 judging criterion leaked into the prompt: {criterion}')

smoke_raw, _ = StubGenerator().generate(smoke_system, smoke_user, seed=SEED)
smoke_code = extract_python(smoke_raw)
if not smoke_code:
    raise AssertionError('S.10 smoke pipeline produced no Python code.')

smoke_path = write_module(smoke_code, folder('selftest') / 'situational_S10_stub.py')
smoke_probe = run_probe(smoke_path)
smoke_verdict = mechanical_verdict(smoke_code, smoke_probe)

if smoke_verdict['mechanical_status'] != 'PASS':
    raise AssertionError('S.10 pipeline smoke test failed:\n'
                         + '\n'.join(smoke_verdict['errors']))
if smoke_probe.get('v2') is None:
    raise AssertionError('S.10 smoke test produced no v2 export for the shared evaluation')

(folder('selftest') / 'situational_S10_stub.v2.json').write_text(
    json.dumps(smoke_probe['v2'], indent=2), encoding='utf-8')

print('S.10 end-to-end smoke test: PASSED')
print('  source        :', smoke_case['source_file'])
print("  Jobst validate:", smoke_probe['jobst_validate']['passed'])
print('  v2 export     :', len(smoke_probe['v2']['states']), 'states,',
      len(smoke_probe['v2']['players']), 'players')
for note in smoke_probe['v2_notes']:
    print('  [v2 note]', note)
print('  semantic correctness was NOT asserted')


S.10 end-to-end smoke test: PASSED
  source        : data/Test_Cases/02_situational/situational_S_10_Test_Case_S10.md
  Jobst validate: True
  v2 export     : 5 states, 2 players
  [v2 note] observations marginalised out of the transition kernel (POSG -> stochastic game)
  [v2 note] player 'type' (human/robot) omitted: Jobst's ABC identifies players by index only
  semantic correctness was NOT asserted


## G. Source inspection, Drive persistence, attempts, and optional repair

Static source inspection records what the model wrote but does not judge whether a probability is justified.

Every completed attempt is appended to `records.jsonl` on Google Drive. `RESUME_RUN_ID` can point to an existing Drive run so completed attempt tags are reused rather than regenerated.

The optional repair receives only concrete mechanical errors and runs only after **FAIL**. `INCOMPLETE` is not repaired because a validator cap is not itself a model error.

In [ ]:
PROBABILITY_NAME = re.compile(
    r'(^p_|prob|probability|chance|likelihood|prior|rate)', re.I
)


def source_inspection(code_text: str) -> dict:
    """Static read of generated source. Records features; never corrects them."""
    info = {
        'module_assignments': [],
        'probability_like_assignments': [],
        'kernel_numeric_literals': [],
        'fractional_literals_all_methods': [],
        'is_terminal_source': None,
        'n_lines': len(code_text.splitlines()) if code_text else 0,
        'parse_error': None,
    }
    if not code_text:
        return info

    try:
        tree = ast.parse(code_text)
    except SyntaxError as exc:
        info['parse_error'] = str(exc)
        return info

    for node in tree.body:
        if not isinstance(node, ast.Assign):
            continue
        value_source = ast.unparse(node.value)
        for target in node.targets:
            if not isinstance(target, ast.Name):
                continue
            assignment = f'{target.id} = {value_source}'
            info['module_assignments'].append(assignment)
            if PROBABILITY_NAME.search(target.id):
                info['probability_like_assignments'].append(assignment)

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == 'transition_kernel':
            for sub in ast.walk(node):
                if (
                    isinstance(sub, ast.Constant)
                    and isinstance(sub.value, (int, float))
                    and not isinstance(sub.value, bool)
                    and 0 < sub.value < 1
                ):
                    info['kernel_numeric_literals'].append(sub.value)

        if isinstance(node, ast.FunctionDef) and node.name == 'is_terminal':
            info['is_terminal_source'] = (
                ast.get_source_segment(code_text, node) or ast.unparse(node)
            )

        if (
            isinstance(node, ast.Constant)
            and isinstance(node.value, (int, float))
            and not isinstance(node.value, bool)
            and 0 < node.value < 1
        ):
            info['fractional_literals_all_methods'].append(node.value)

    return info


def attempt_tag(case: dict, spec, variant: str, generation: int, repair_round: int) -> str:
    """Unique per (case, MODEL, variant, generation, repair round).

    The model slug is part of the tag. Without it the resume cache in records.jsonl
    collides across models: the second model's first attempt would be served from the
    first model's cached record, and the multi-model comparison would silently become
    five copies of one model's results.
    """
    return (f"{case['case_id'].replace('.', '')}_{spec.slug}_{variant}"
            f"_g{generation}_r{repair_round}")


def is_reusable(record: dict) -> tuple[bool, str]:
    """Whether a cached attempt is a RESULT, or an artefact of a setting we have changed.

    The resume cache exists so an interrupted run is not paid for twice. It must never
    make a *harness* failure permanent: if a reply was cut off by our token budget, or
    never reached a model at all, reusing it would freeze the bug into the results and
    the fix would silently do nothing.

    So a non-PASS record is reused only when the model actually produced code and the
    reply was not truncated. A genuine modelling failure -- code that does not implement
    the interface, distributions that do not sum to 1 -- is a real result and is kept.
    """
    meta = record.get('meta') or {}
    verdict = record.get('verdict') or {}

    if meta.get('error'):
        return False, 'the call never reached a model'
    if verdict.get('mechanical_status') == 'PASS':
        return True, ''
    if meta.get('truncated'):
        return False, 'the reply was cut off by our token budget'
    if not record.get('code_generated'):
        return False, 'the model returned no usable code (budget or reasoning setting)'
    return True, ''


def load_existing_attempts(path: Path) -> dict:
    existing, discarded = {}, {}
    if not path.exists():
        return existing
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue        # ignore only a partially written final line
            if not record.get('tag'):
                continue
            reusable, why = is_reusable(record)
            if reusable:
                existing[record['tag']] = record
                discarded.pop(record['tag'], None)
            else:
                existing.pop(record['tag'], None)
                discarded[record['tag']] = why

    if discarded:
        print(f'resume cache: {len(discarded)} cached attempt(s) will be RE-RUN, not '
              f'reused, because they were caused by settings rather than by the model:')
        for reason in sorted(set(discarded.values())):
            slugs = sorted({tag.split("_")[1] for tag, why in discarded.items()
                            if why == reason})
            n = sum(1 for why in discarded.values() if why == reason)
            print(f'  {n:>4}  {reason}  [{", ".join(slugs)}]')
        print('  Nothing needs deleting from Drive by hand: the old files are simply '
              'overwritten when each attempt is re-run.')
    return existing


EXISTING_ATTEMPTS = load_existing_attempts(RECORDS_PATH)
print(f'resume cache: {len(EXISTING_ATTEMPTS)} attempt(s) will be reused')


def append_record(attempt: dict) -> None:
    record = {key: value for key, value in attempt.items() if key != 'raw'}
    with RECORDS_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, default=str) + '\n')
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass


def run_attempt(case: dict, spec, variant: str, generation: int,
                repair_round: int, errors: str = '') -> dict:
    tag = attempt_tag(case, spec, variant, generation, repair_round)
    system, user = build_prompt(case, variant)
    if repair_round:
        user += '\n\n' + REPAIR_INSTRUCTION.format(errors=errors[:3000])

    seed = SEED + 1000 * generation + repair_round

    try:
        raw, meta = generator_for(spec).generate(system, user, seed)
    except Exception as exc:
        # No credit, bad token, forbidden model: nothing downstream will fix it, and
        # continuing would spend the rest of the grid writing records that never
        # reached a model. Stop immediately and say what to do.
        if MODELREG.is_fatal(exc):
            raise MODELREG.FatalBackendError(
                f'{type(exc).__name__}: {exc}\n\n'
                f'Stopped at {tag} -- the backend cannot serve further requests.\n'
                f'Nothing is lost: completed attempts are in records.jsonl and will be\n'
                f'reused. Fix the account (credits / token / model access), then set\n'
                f'   RESUME_RUN_ID = {RUN_ID!r}\n'
                f'in cell 3 and re-run from there.'
            ) from exc
        raw = ''
        meta = {
            'backend': spec.backend, 'model_slug': spec.slug, 'model_id': spec.repo_id,
            'model_revision': None, 'input_tokens': None, 'output_tokens': None,
            'truncated': None, 'seconds': None, 'seed': seed,
            'error': f'{type(exc).__name__}: {exc}',
        }

    extraction = extract_result(raw)
    generated_code = extraction.code

    # All artefacts below are written to DRIVE only.
    (folder('prompts') / f'{tag}.md').write_text(
        '# SYSTEM\n\n' + system + '\n\n# USER\n\n' + user + '\n', encoding='utf-8')
    (folder('raw') / f'{tag}.txt').write_text(raw or '', encoding='utf-8')

    module_path = None
    # Say WHICH kind of nothing came back. "no code in the response" hid a budget spent
    # entirely on a reasoning trace behind the same label as a flat refusal.
    no_code = EXTRACT.NO_CODE_REASONS.get(extraction.reason, 'no code in the response')
    probe = {'stage': extraction.reason or 'no_code', 'status': 'FAIL',
             'errors': [no_code],
             'jobst_validate': {'passed': False, 'error': extraction.reason or 'no code'},
             'exploration': None, 'v2': None, 'v2_notes': []}
    if generated_code:
        module_path = write_module(generated_code, folder('code') / f'{tag}.py')
        probe = run_probe(module_path)

    (folder('probe') / f'{tag}.json').write_text(
        json.dumps(probe, indent=2, default=str), encoding='utf-8')

    # THE HANDOFF: every mechanically valid model is also written in the v2 schema,
    # which is what json_generation/lane1.py consumes. This is what Souhaila receives.
    # First passes and repaired models go to SEPARATE folders. A first pass is what the
    # LLM produced; a repaired one is what it produced after our harness told it what was
    # wrong -- a different object, and one the JSON track never produces. Mixing them in
    # one folder would let a downstream glob silently compare unlike things.
    v2_path = None
    if probe.get('v2'):
        v2_path = folder('first_pass' if repair_round == 0 else 'repaired') / f'{tag}.json'
        v2_path.write_text(json.dumps({**probe['v2'],
                                       '_provenance': {
                                           'tag': tag,
                                           'case_id': case['case_id'],
                                           'model': spec.repo_id,
                                           'prompt_variant': variant,
                                           'generation': generation,
                                           'repair_round': repair_round,
                                           'source_commit_sha': SOURCE_COMMIT_SHA,
                                           'lossy_conversion_notes': probe.get('v2_notes', []),
                                       }}, indent=2), encoding='utf-8')

    verdict = mechanical_verdict(generated_code, probe)
    if not generated_code and meta.get('error'):
        verdict['mechanical_status'] = 'FAIL'
        verdict['mechanical_ok'] = False
        verdict['failed_stage'] = 'backend_error'
        verdict['errors'] = [meta['error']]

    attempt = {
        'case_id': case['case_id'],
        'title': case['title'],
        'source_file': case['source_file'],
        'model_slug': spec.slug,
        'model_label': spec.label,
        'prompt_variant': variant,
        'generation': generation,
        'repair_round': repair_round,
        'stage': 'repair' if repair_round else 'first_pass',
        'tag': tag,
        'code_generated': bool(generated_code),
        'no_code_reason': extraction.reason or None,
        'had_reasoning_trace': extraction.had_reasoning,
        'reasoning_unclosed': extraction.reasoning_unclosed,
        'prose_before_code': extraction.prose_before_code,
        'code': generated_code,
        'raw': raw,
        'meta': meta,
        'probe': probe,
        'verdict': verdict,
        'inspection': source_inspection(generated_code),
        'module_path': str(module_path.relative_to(RUN_DIR)) if module_path else None,
        'v2_path': str(v2_path.relative_to(RUN_DIR)) if v2_path else None,
        'semantic_status': 'pending_human_review',
    }
    append_record(attempt)
    return attempt


def get_or_run_attempt(case: dict, spec, variant: str, generation: int,
                       repair_round: int, errors: str = '') -> dict:
    tag = attempt_tag(case, spec, variant, generation, repair_round)
    if tag in EXISTING_ATTEMPTS:
        print('resume:', tag)
        return EXISTING_ATTEMPTS[tag]
    attempt = run_attempt(case, spec, variant, generation, repair_round, errors=errors)
    EXISTING_ATTEMPTS[tag] = {k: v for k, v in attempt.items() if k != 'raw'}
    return attempt


def run_cell(case: dict, spec, variant: str, generation: int) -> list:
    """One grid cell: the first pass, then repair rounds until it passes or we stop.

    Each round is shown the errors from the round before it, so the model is correcting
    its own latest attempt rather than re-reading the original failure. Repairs stop
    early on PASS, and are never attempted when no code was produced at all -- there is
    nothing to repair, and the failure is the backend's or the extractor's.
    """
    first = get_or_run_attempt(case, spec, variant, generation, repair_round=0)
    attempts = [first]
    if not (ENABLE_REPAIR and first['code_generated']):
        return attempts

    latest = first
    for repair_round in range(1, MAX_REPAIR_ROUNDS + 1):
        if latest['verdict']['mechanical_status'] != 'FAIL':
            break
        latest = get_or_run_attempt(
            case, spec, variant, generation, repair_round=repair_round,
            errors='\n'.join(latest['verdict']['errors']))
        attempts.append(latest)
        if not latest['code_generated']:
            break
    return attempts


resume cache: 4 cached attempt(s) will be RE-RUN, not reused, because they were caused by settings rather than by the model:
     4  the reply was cut off by our token budget  [claude-opus-5, deepseek-v4-flash]
  Nothing needs deleting from Drive by hand: the old files are simply overwritten when each attempt is re-run.
resume cache: 69 attempt(s) will be reused


## H. Run

The grid is **cases × models × prompt variants × generations**, with one mechanical
repair round attempted on every first-pass `FAIL`.

Attempt tags carry the model slug, so the resume cache in `records.jsonl` is keyed per
model — re-running after adding a model regenerates only the new cells, and two models
can never collide on the same tag.

With `SMOKE_TEST_ONLY = True` the grid collapses to one case × one variant × one
generation per model: enough to prove the wiring and measure per-model latency before
committing to the full run.


In [ ]:
if SMOKE_TEST_ONLY:
    GRID = [(CASES[0], spec, STRATEGY, 0) for spec in AVAILABLE_MODELS]
    print('SMOKE TEST: one case x one generation per model.')
    print('Set SMOKE_TEST_ONLY = False for the full grid.\n')
else:
    GRID = [
        (case, spec, variant, generation)
        for case in CASES
        for spec in AVAILABLE_MODELS
        for variant in PROMPT_VARIANTS
        for generation in range(N_GENERATIONS)
    ]
    print(
        f'{len(CASES)} cases x {len(AVAILABLE_MODELS)} models x '
        f'{len(PROMPT_VARIANTS)} variants x {N_GENERATIONS} generations '
        f'= {len(GRID)} first-pass generations'
        + (' (+ FAIL-only mechanical repairs)' if ENABLE_REPAIR else '')
    )

ATTEMPTS = []
started = time.time()
failures_by_model = {}

STOPPED_EARLY = None
for index, (case, spec, variant, generation) in enumerate(GRID, start=1):
    try:
        cell_attempts = run_cell(case, spec, variant, generation)
    except MODELREG.FatalBackendError as exc:
        STOPPED_EARLY = str(exc)
        print(f'\n{"=" * 78}\nRUN STOPPED at {index}/{len(GRID)}\n{"=" * 78}\n{exc}\n')
        break
    for attempt in cell_attempts:
        ATTEMPTS.append(attempt)
        verdict = attempt['verdict']
        meta = attempt['meta']
        if verdict['mechanical_status'] != 'PASS' and attempt['repair_round'] == 0:
            failures_by_model[spec.slug] = failures_by_model.get(spec.slug, 0) + 1
        note = ''
        if not attempt['code_generated']:
            # The single most useful thing to see live: an empty reply that is really a
            # truncated reasoning trace needs a bigger MAX_NEW_TOKENS, not a better model.
            note = f' [{attempt.get("no_code_reason") or "no_code"}]'
            if attempt.get('reasoning_unclosed'):
                note += ' RAISE MAX_NEW_TOKENS'
        elif meta.get('truncated'):
            note = ' [truncated]'
        print(
            f'[{index:>4}/{len(GRID)}] {attempt["tag"]:<42} '
            f'code={"y" if attempt["code_generated"] else "n"}{note} '
            f'exec={"y" if verdict["exec_ok"] else "n"} '
            f'val={"y" if verdict["validate_ok"] else "n"} '
            f'mech={verdict["mechanical_status"]:<10} '
            f'{str(meta.get("seconds", "")):>7}s '
            f'{(verdict["errors"] or [""])[0][:56]}'
        )

elapsed = (time.time() - started) / 60
print(f'\n{len(ATTEMPTS)} attempts in {elapsed:.1f} min'
      + ('  (RUN INCOMPLETE -- see the stop message above)' if STOPPED_EARLY else ''))
if failures_by_model:
    print('first-pass non-PASS by model:',
          ', '.join(f'{k}={v}' for k, v in sorted(failures_by_model.items())))

# A model that failed on every single attempt is usually a config problem (wrong repo
# id, a provider that rejects the request), not a capability result. Say so loudly
# rather than letting a row of zeros reach the paper.
for spec in AVAILABLE_MODELS:
    rows = [a for a in ATTEMPTS if a['meta'].get('model_slug') == spec.slug
            and a['repair_round'] == 0]
    if rows and all(a['meta'].get('error') for a in rows):
        print(
            f'\n  WARNING: every call to {spec.slug} raised a backend error. '
            f'This is a configuration failure, not a score of zero. '
            f'First error: {rows[0]["meta"].get("error")}'
        )


11 cases x 5 models x 1 variants x 3 generations = 165 first-pass generations (+ FAIL-only mechanical repairs)
resume: S1_qwen3.8-27b_v2_python_g0_r0
[   1/165] S1_qwen3.8-27b_v2_python_g0_r0             code=y exec=y val=y mech=PASS        184.27s 
resume: S1_qwen3.8-27b_v2_python_g1_r0
[   2/165] S1_qwen3.8-27b_v2_python_g1_r0             code=y exec=y val=y mech=PASS         301.0s 
resume: S1_qwen3.8-27b_v2_python_g2_r0
resume: S1_qwen3.8-27b_v2_python_g2_r1
resume: S1_qwen3.8-27b_v2_python_g2_r2
[   3/165] S1_qwen3.8-27b_v2_python_g2_r0             code=y exec=n val=n mech=FAIL         267.1s no POSG subclass was defined
[   3/165] S1_qwen3.8-27b_v2_python_g2_r1             code=y exec=n val=n mech=FAIL        222.35s no POSG subclass was defined
[   3/165] S1_qwen3.8-27b_v2_python_g2_r2             code=y exec=y val=y mech=PASS        239.08s 
resume: S1_deepseek-v4-flash_v2_python_g0_r0
[   4/165] S1_deepseek-v4-flash_v2_python_g0_r0       code=y exec=y val=y mech=PASS       129

config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

... (4 outputs left out)

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

loaded in 168.7s | revision: None | 50.1 GB on the GPU
[  61/165] S5_qwen3.8-27b_v2_python_g0_r0             code=y exec=y val=y mech=PASS         78.82s 
[  62/165] S5_qwen3.8-27b_v2_python_g1_r0             code=y exec=y val=y mech=PASS         83.53s 
[  63/165] S5_qwen3.8-27b_v2_python_g2_r0             code=y exec=y val=y mech=PASS        101.86s 
[  64/165] S5_deepseek-v4-flash_v2_python_g0_r0       code=y exec=n val=n mech=FAIL        133.12s NameError: name 'Generic' is not defined
[  64/165] S5_deepseek-v4-flash_v2_python_g0_r1       code=y exec=y val=y mech=PASS         85.86s 
[  65/165] S5_deepseek-v4-flash_v2_python_g1_r0       code=y exec=y val=y mech=PASS        761.71s 
[  66/165] S5_deepseek-v4-flash_v2_python_g2_r0       code=y exec=y val=y mech=PASS       1089.61s 
[  67/165] S5_gpt-5.6-sol_v2_python_g0_r0             code=y exec=y val=y mech=PASS         33.56s 
[  68/165] S5_gpt-5.6-sol_v2_python_g1_r0             code=y exec=y val=y mech=PASS         43.06s 
[  69

## I. Results

All tables and figures come from `python_prompting/analysis.py`, `figures.py` and
`tables.py` in the pinned clone, so the numbers in the paper are tied to
`SOURCE_COMMIT_SHA` and not to anything pasted into a cell. That code is unit-tested
offline against synthetic records (`tests/test_pipeline.py`).

Two statistical choices are deliberate and belong in the paper's methods section:

**Wilson intervals, not normal approximations.** A per-model cell holds `11 × k`
observations — 33 at `k = 3`. The normal approximation misbehaves at that size and
produces intervals crossing 0 or 1 exactly where these models sit.

**The case is the resampling unit.** Three generations of one situation are not three
independent draws, and the eleven situations differ far more from one another than
repeated draws of one situation do. Model comparisons therefore bootstrap over **cases**.
Treating rows as independent would shrink every interval and overstate every difference.

`INCOMPLETE` is never folded into `FAIL`: it means the mechanical sweep hit an
exploration cap, so the model is *unjudged*, not wrong. Only `PASS` counts as success.

Every headline number is read from **first-pass** attempts. Repairs are reported
separately — folding them into the headline would measure the harness, not the model, and
the JSON track does not repair at all.

With one prompt strategy, the prompt-ablation tables and figures are empty by
construction and are skipped rather than drawn. The `prompt_variant` column stays on
every frame so a later run can add an arm without touching the analysis code.


In [ ]:
ATTEMPTS_DF = ANALYSIS.attempts_frame(ATTEMPTS)
FIRST_PASS_DF = ANALYSIS.first_pass(ATTEMPTS_DF)

print(f'{len(ATTEMPTS_DF)} attempts  |  {len(FIRST_PASS_DF)} first-pass  |  '
      f'{FIRST_PASS_DF["model_slug"].nunique()} models  |  '
      f'{FIRST_PASS_DF["case_id"].nunique()} cases')

if FIRST_PASS_DF.empty:
    raise RuntimeError('No first-pass attempts to analyse.')

# Paired contrasts are the expensive part; compute once and reuse everywhere.
PAIRED = ANALYSIS.paired_model_comparison(
    FIRST_PASS_DF, n_boot=BOOTSTRAP_DRAWS, confidence=CONFIDENCE, seed=SEED
)
VARIANT_EFFECT = ANALYSIS.variant_effect(
    FIRST_PASS_DF, n_boot=BOOTSTRAP_DRAWS, confidence=CONFIDENCE, seed=SEED
)

DERIVED = {
    'rates': ANALYSIS.rate_table(FIRST_PASS_DF, ['model_slug'], confidence=CONFIDENCE),
    'rates_by_variant': ANALYSIS.rate_table(
        FIRST_PASS_DF, ['model_slug', 'prompt_variant'], confidence=CONFIDENCE),
    'status': ANALYSIS.status_breakdown(FIRST_PASS_DF, ['model_slug']),
    'funnel': ANALYSIS.funnel_table(FIRST_PASS_DF),
    'taxonomy': ANALYSIS.failure_taxonomy(FIRST_PASS_DF),
    'difficulty': ANALYSIS.case_difficulty(FIRST_PASS_DF),
    'pass_at_k': ANALYSIS.pass_at_k(FIRST_PASS_DF),
    'consistency': ANALYSIS.self_consistency(FIRST_PASS_DF),
    'structure': ANALYSIS.structure_table(FIRST_PASS_DF),
    'cost': ANALYSIS.cost_table(FIRST_PASS_DF),
    'repair': ANALYSIS.repair_lift(ATTEMPTS_DF),
    'paired': PAIRED,
    'variant_effect': VARIANT_EFFECT,
    'headline': ANALYSIS.headline(FIRST_PASS_DF),
    # Keyed to the project's own criteria for a world model the power-estimation
    # algorithms could actually use -- not just a well-formed POSG.
    'criteria': ANALYSIS.criteria_table(FIRST_PASS_DF),
    'agency': ANALYSIS.agency_table(FIRST_PASS_DF),
    'coverage': ANALYSIS.handoff_coverage(FIRST_PASS_DF),
    'sensitivity': ANALYSIS.design_sensitivity(
        FIRST_PASS_DF, paired_tables={'models': PAIRED, 'variants': VARIANT_EFFECT}
    ),
}

# Everything to Drive: the raw frames as CSV, so the analysis can be redone offline.
ATTEMPTS_DF.to_csv(folder('derived') / 'all_attempts.csv', index=False)
FIRST_PASS_DF.to_csv(folder('derived') / 'first_pass_only.csv', index=False)
for name, table in DERIVED.items():
    if isinstance(table, pd.DataFrame) and not table.empty:
        table.to_csv(folder('derived') / f'{name}.csv', index=(name == 'taxonomy'))

print('derived tables:', ', '.join(sorted(DERIVED)))


In [ ]:
print('HEADLINE — mechanical validity by model (first pass)')
display(DERIVED['headline'][[
    'model_slug', 'n', 'k', 'rate', 'ci_low', 'ci_high',
    'pass_at_1', 'pass_at_k', 'INCOMPLETE', 'verdict_agreement',
    'stochastic_pct', 'med_output_tokens', 'med_seconds',
]].round(3))

print('\nARE THE VALID MODELS USABLE WORLD MODELS? (share of valid outputs)')
display(DERIVED['criteria'].round(3))
print(
    'Mechanical validity only says the LLM produced a well-formed POSG. A single-step, '
    'deterministic, fully-observed one-player game passes every check in this notebook '
    'and cannot support a power estimate: no humans, nothing uncertain, no later.'
)

print('\nDO THE HUMANS GET TO DO ANYTHING?')
display(DERIVED['agency'].round(2))
print(
    'Read the MAX columns and the choice shares, not the medians. The v2_python prompt '
    'instructs a singleton `noop` for a player that is not acting in a state, so a mean '
    'action width punishes a model for taking turns exactly as we asked. `*_actions_max` '
    'is the widest choice each side ever gets; `*_choice_share` is how much of the model '
    'that covers; `ratio` is the agent\'s widest choice over the humans\'.'
)

print('\nWHAT THERE IS TO HAND OVER (valid models per situation x model)')
display(DERIVED['coverage'])

print('\nPAIRED MODEL COMPARISONS (cluster bootstrap over cases, Holm-corrected)')
display(DERIVED['paired'][[
    'model_a', 'model_b', 'n_paired', 'rate_a', 'rate_b',
    'diff', 'ci_low', 'ci_high', 'p_holm', 'favours',
]].round(3))

if len(DERIVED['variant_effect']):
    print('\nPROMPT ABLATION (paired within case x generation)')
    display(DERIVED['variant_effect'][[
        'model_slug', 'prompt_variant', 'n_paired', 'rate_baseline', 'rate_variant',
        'diff', 'ci_low', 'ci_high', 'p_holm', 'decisive_uncorrected', 'decisive',
    ]].round(3))
    print(f'\nprompt effects significant before correction: '
          f'{int(DERIVED["variant_effect"]["decisive_uncorrected"].sum())}'
          f'  |  after Holm across the family: '
          f'{int(DERIVED["variant_effect"]["decisive"].sum())}')
else:
    print(f'\nNo prompt ablation: this run uses one strategy, {STRATEGY!r}, matching '
          f'the JSON track. See section D for why.')

print('\nDESIGN SENSITIVITY — what this grid could have detected')
display(DERIVED['sensitivity'].round(3))
print(
    'Read the paired rows: if the MDE exceeds the effect you care about, a null result '
    'is a statement about the design, not about prompting. Report this next to any '
    'null finding.'
)

print('\nWHERE MODELS DROP OUT')
display(DERIVED['funnel'].pivot_table(
    index='model_slug', columns='label', values='rate').round(3))

print('\nFAILURE TAXONOMY (counts)')
display(DERIVED['taxonomy'])

print('\nSITUATION DIFFICULTY (hardest first)')
display(DERIVED['difficulty'][[
    'case_id', 'title', 'n', 'k', 'rate', 'ci_low', 'ci_high', 'n_models_any_pass'
]].round(3))

if not DERIVED['repair'].empty:
    print('\nREPAIR LIFT (first-pass FAILs recovered by one round)')
    display(DERIVED['repair'].round(3))


### I.1 Figures

Sixteen figures, written to `<run>/figures/` as both PNG (notebook, slides) and PDF
(vector, for LaTeX `\includegraphics`).

Two of them — `f04_model_variant_heatmap` and `f05_variant_effect` — draw a prompt
ablation, and this run has one prompt strategy. They are **skipped by name with a
reason**, so their absence is on the record rather than looking like a rendering failure.
Fourteen figures are produced.

The palette is the project's validated categorical set: it clears the lightness band,
chroma floor, adjacent-pair colour-vision-deficiency separation and the normal-vision
floor on the chart surface. Each model keeps **one hue in every figure**, assigned by
identity and never by rank, so a model that moves in the sorting does not change colour
between plots. Some hues sit below 3:1 contrast against the surface, so every figure
carries direct value labels and every figure has a table counterpart — identity is never
carried by colour alone.


In [ ]:
FIGURE_PATHS = FIGURES.render_all(
    FIRST_PASS_DF, DERIVED, folder('figures'), formats=('png', 'pdf')
)
rendered = {name: paths for name, paths in FIGURE_PATHS.items() if paths}
print(f'{len(rendered)}/{len(FIGURE_PATHS)} figures written to {folder("figures")}')
for name in sorted(rendered):
    print('  ', name)

TABLE_PATHS = TABLES.write_paper_tables(DERIVED, folder('latex'), MODEL_LABELS)
print(f'\n{len(TABLE_PATHS)} LaTeX tables written to {folder("latex")}')
for name in sorted(TABLE_PATHS):
    print('   \\input{tables/' + name + '}')

# Show the headline figures inline.
from IPython.display import Image  # noqa: E402
for name in ('f13_criteria', 'f01_pass_rate', 'f02_funnel', 'f14_horizon',
             'f15_agency', 'f16_handoff_coverage', 'f04_model_variant_heatmap'):
    if rendered.get(name):
        display(Image(filename=str(folder('figures') / f'{name}.png')))


### I.2 The handoff to the shared evaluation

Every mechanically valid model is written twice:

- `generated/<tag>.py` — the Python the model actually produced, against Jobst's ABC.
- `v2/<tag>.json` — the same model rolled out into the project's **v2 schema**, which is
  what `json_generation/lane1.py` consumes directly.

Two things are lost in the v2 conversion, recorded in `_provenance.lossy_conversion_notes`
on every file rather than left implicit:

1. **Observations are marginalised out.** Jobst's kernel is `P(s′, o | s, a)`; v2's
   `next` holds only `{state, p}`. Summing over `o` is exactly the POSG-to-stochastic-game
   reduction — it discards what makes this a POSG.
2. **Player `type` (human/robot) is omitted.** Jobst's ABC identifies players by index
   only; there is nothing to map onto that field, and it is the field that decides whose
   power the downstream analysis measures. It is left absent rather than guessed.

Only `PASS` models are exported. A capped or failing exploration would produce states
that are neither terminal nor fully expanded, and those would read as modelling errors
rather than as a budget we imposed.


In [ ]:
first_pass_v2 = sorted(folder('first_pass').glob('*.json'))
repaired_v2 = sorted(folder('repaired').glob('*.json'))
print(f'{len(first_pass_v2)} first-pass model(s)  |  {len(repaired_v2)} repaired model(s)')
print(
    "The DELIVERABLE is v2/first_pass -- what the models produced unaided. The JSON "
    "track does not repair at all (generate.py: 'Nothing is stripped or repaired ... "
    "judges the reply as it came back'), so only first passes are comparable across "
    "the two tracks. v2/repaired is a separate result about the value of feedback, "
    "and must be labelled as such wherever it is used."
)

# Verify locally that what we hand over actually passes the canonical checker.
sys.path.insert(0, str(REPO_ROOT / 'json_generation'))
import lane1 as LANE1  # noqa: E402

rows = []
for path in first_pass_v2 + repaired_v2:
    model = json.loads(path.read_text())
    provenance = model.pop('_provenance', {})
    result = LANE1.run_lane1(model, path.stem)
    rows.append({
        'tag': path.stem,
        'case_id': provenance.get('case_id'),
        'model': provenance.get('model'),
        'prompt_variant': provenance.get('prompt_variant'),
        'repair_round': provenance.get('repair_round', 0),
        'bucket': 'first_pass' if provenance.get('repair_round', 0) == 0 else 'repaired',
        'lane1_verdict': result['verdict'],
        'lane1_fails': result['n_fail'],
        'first_finding': str(result['findings'][0])[:160] if result['findings'] else '',
    })

if rows:
    handoff_df = pd.DataFrame(rows)
    handoff_df.to_csv(folder('first_pass').parent / 'handoff_lane1_check.csv',
                      index=False)
    display(handoff_df.groupby('bucket')['lane1_verdict'].value_counts().unstack(fill_value=0))

    deliverable = handoff_df[handoff_df['bucket'] == 'first_pass']
    n_pass = int((deliverable['lane1_verdict'] == 'PASS').sum())
    print(f'\n{n_pass}/{len(deliverable)} FIRST-PASS models pass lane1.py')
    if n_pass < len(deliverable):
        print('Models that pass our probe but fail lane1 are worth reporting: they mark '
              'exactly where the Python and JSON formalisms disagree.')

    print(f'\nSend Souhaila: {folder("first_pass")}'
          f'\n  plus {folder("first_pass").parent / "handoff_lane1_check.csv"}')
    if repaired_v2:
        print(f'Optional, and only if labelled repaired: {folder("repaired")}')
else:
    print('No models to hand over yet.')


### I.3 State-transition graphs

The figures above are aggregate — rates and distributions across models and cases. These
show what an individual model actually *built*: the reachable state graph of a single
generated POSG.

They follow the game track's visual language (`report/generate_game_eda_report.py`), so
the two tracks' figures read as one set: blue initial state, orange terminal, parallel
transitions aggregated onto a single edge labelled with the joint-action count, or with
the probability where there is only one.

Two things differ, because situational models differ from the game references:

- **Laid out by depth.** The game cases are one step deep; situational models run
  several, so states are placed by discovered depth (which reduces to the game layout
  for a one-step model), with one barycentre pass to cut edge crossings.
- **Labels are the state tuple itself.** Jobst's ABC carries no state descriptions, so
  there is nothing to print but each state's `repr`. That is worth seeing in its own
  right: a model whose states are bare integers has represented the situation very
  differently from one whose states read `(0, 'unresolved', 'silent')`.

The **per-case comparison** is the one to look at first. One panel per model, same
situation — it answers "did the models even build the same kind of world model?", which
no pass rate can show.

Graphs beyond `MAX_DRAWABLE_STATES` states are not drawn; the panel says so rather than
emitting an unreadable hairball, and the numbers still reach the tables.


In [ ]:
from python_prompting import graphs as GRAPHS  # noqa: E402

GRAPH_DIR = folder('graphs')

valid_attempts = [
    a for a in ATTEMPTS
    if a['repair_round'] == 0
    and a['verdict']['mechanical_status'] == 'PASS'
    and (a.get('probe') or {}).get('exploration')
]
print(f'{len(valid_attempts)} mechanically valid first-pass models available to draw')

# --- one graph per (case, model): the earliest generation ---------------------------
def _pick(case_id, slug):
    """The attempt we draw for this (case, model): the earliest generation that passed.

    With one prompt strategy the only tie-break left is the generation index, and taking
    the first is the only choice that does not select on the outcome.
    """
    candidates = [a for a in valid_attempts
                  if a['case_id'] == case_id and a['model_slug'] == slug]
    if not candidates:
        return None
    return min(candidates, key=lambda a: a['generation'])

drawn = 0
for case in CASES:
    for spec in AVAILABLE_MODELS:
        attempt = _pick(case['case_id'], spec.slug)
        if attempt is None:
            continue
        GRAPHS.draw_state_graph(
            attempt['probe']['exploration'],
            f"{case['case_id']} — {case['title']}",
            GRAPH_DIR / f"{attempt['tag']}.png",
            subtitle=(f"{spec.label} · {attempt['prompt_variant']} · "
                      f"generation {attempt['generation']}"),
        )
        drawn += 1

# Optionally draw every valid attempt, not just one per (case, model).
if GRAPH_ALL_ATTEMPTS:
    for attempt in valid_attempts:
        path = GRAPH_DIR / f"{attempt['tag']}.png"
        if path.exists():
            continue
        case = next(c for c in CASES if c['case_id'] == attempt['case_id'])
        GRAPHS.draw_state_graph(
            attempt['probe']['exploration'],
            f"{case['case_id']} — {case['title']}",
            path,
            subtitle=(f"{attempt.get('model_label') or attempt['model_slug']} · "
                      f"{attempt['prompt_variant']} · generation {attempt['generation']}"),
        )
        drawn += 1

print(f'{drawn} individual state graphs written to {GRAPH_DIR}')

# --- one comparison figure per case: every model on the same situation --------------
comparison_paths = []
for case in CASES:
    entries = []
    for spec in AVAILABLE_MODELS:
        attempt = _pick(case['case_id'], spec.slug)
        entries.append((spec.label,
                        attempt['probe']['exploration'] if attempt else None))
    path = GRAPH_DIR / f"comparison_{case['case_id'].replace('.', '')}.png"
    GRAPHS.draw_case_comparison(entries, case['case_id'], case['title'], path)
    comparison_paths.append(path)

print(f'{len(comparison_paths)} per-case comparison figures written')

# Show the comparison for the first case inline.
if comparison_paths and comparison_paths[0].exists():
    display(Image(filename=str(comparison_paths[0])))


### I.4 Graph EDA — Gagana's `world_model_eda`, on the situational models

`world_model_eda/eda.py` is the project's shared graph analysis: branching factor,
transition entropy, degree, density, modularity, distance to a terminal state, and six
plots over them. It was written against the **game** test cases, whose world models
arrive as JSON.

`python_prompting/eda_bridge.py` feeds this track's models into the same code, so both
tracks are described by the same statistics and the paper can put them side by side
without a footnote about differing definitions. Nothing about the analysis is
reimplemented here — every number and every plot is her method, called on her object.

**What crosses the bridge:** the reachable state set, terminality, the joint actions
available in each state, every transition outcome and its probability, and the opening
state.

**What does not:** observations and rewards. Her `Transition` is
`(state, joint action, next state, probability)` — so these panels are a view of the
*dynamics* of a POSG, which is exactly what they are for the JSON track too, since that
schema has no observations at all. Partial observability is measured elsewhere in this
notebook, by the probe's `has_hidden_state` and `observations_differ_by_player` signals
and by the criteria figure (F13).

Two pages per generated model, plus a cross-model table:

- **`*_eda_dist.png`** — branching factor, transition probabilities, transition entropy,
  degree. These are the ones to read: they say what *kind* of world model each LLM built,
  not merely whether it ran.
- **`*_eda_graphs.png`** — her state graph and state-action graph at full size. Her state
  graph writes every joint action onto the edge it belongs to, which is informative for a
  small model and crowded for a large one; the readable per-case figure for the paper is
  the one in I.3 above, which aggregates parallel edges to a count. Both are kept: hers
  is the shared convention, mine is the legible one.
- **`eda_summary.csv`** — fourteen statistics per generated model, so "did the models
  build the same kind of object?" becomes a table rather than an impression.

States are addressed by the probe's short ids (`s0`, `s1`, ...) because a spring layout
labelled with 40-character tuples is unreadable; `<run>/eda/<tag>_states.csv` maps each
id back to the model's own state value.


In [ ]:
from python_prompting import eda_bridge as EDA  # noqa: E402

EDA_DIR = folder('eda')

eda_rows = []
eda_pages = []

if RUN_WORLD_MODEL_EDA:
    to_describe = valid_attempts if EDA_ALL_ATTEMPTS else [
        attempt for case in CASES for spec in AVAILABLE_MODELS
        if (attempt := _pick(case['case_id'], spec.slug)) is not None
    ]
    print(f'{len(to_describe)} generated models going through world_model_eda')

    for attempt in to_describe:
        exploration = attempt['probe']['exploration']
        label = attempt.get('model_label') or attempt['model_slug']
        title = f"{attempt['case_id']} — {attempt['title']}"
        subtitle = f"{label} · {attempt['prompt_variant']} · generation {attempt['generation']}"
        note = EDA.CAP_WARNING if exploration.get('caps_hit') else ''

        try:
            model = EDA.to_eda(exploration, REPO_ROOT)
        except Exception as exc:
            print(f'  {attempt["tag"]}: skipped ({type(exc).__name__}: {exc})')
            continue

        eda_pages.append(
            EDA.draw_distributions(model, title, EDA_DIR / f'{attempt["tag"]}_eda_dist.png',
                                   subtitle=subtitle, note=note))
        eda_pages.append(
            EDA.draw_graphs(model, title, EDA_DIR / f'{attempt["tag"]}_eda_graphs.png',
                            subtitle=subtitle, note=note))
        pd.DataFrame(EDA.state_legend(exploration)).to_csv(
            EDA_DIR / f'{attempt["tag"]}_states.csv', index=False)

        try:
            statistics = EDA.summary_row(model)
        except Exception as exc:
            print(f'  {attempt["tag"]}: statistics unavailable ({type(exc).__name__}: {exc})')
            statistics = {}
        eda_rows.append({
            'case_id': attempt['case_id'],
            'model': label,
            'model_slug': attempt['model_slug'],
            'generation': attempt['generation'],
            'capped': bool(exploration.get('caps_hit')),
            **statistics,
        })

    EDA_DF = pd.DataFrame(eda_rows)
    if len(EDA_DF):
        EDA_DF.to_csv(EDA_DIR / 'eda_summary.csv', index=False)
        print(f'\n{len(eda_pages)} EDA pages and {len(EDA_DF)} summary rows written to {EDA_DIR}')

        # Per-model medians: one line per LLM, over every situation it modelled.
        numeric = [c for c in EDA_DF.columns
                   if c not in ('case_id', 'model', 'model_slug', 'generation', 'capped')]
        BY_MODEL = (EDA_DF.groupby('model')[numeric]
                    .median(numeric_only=True).round(3).reset_index())
        print('\nGRAPH EDA — median over situations, by model')
        display(BY_MODEL)
        BY_MODEL.to_csv(EDA_DIR / 'eda_by_model.csv', index=False)

        print('\nGRAPH EDA — per situation and model')
        display(EDA_DF.round(3))

        # Show one model's two pages inline.
        for page in eda_pages[:2]:
            display(Image(filename=str(page)))
    else:
        EDA_DF = pd.DataFrame()
        print('no mechanically valid model could be described')
else:
    EDA_DF = pd.DataFrame()
    print('world_model_eda skipped (RUN_WORLD_MODEL_EDA = False)')


In [ ]:
# Exact first-pass mechanical errors, grouped by model.
# INCOMPLETE rows show which exploration cap was hit.
for spec in AVAILABLE_MODELS:
    rows = FIRST_PASS_DF[
        (FIRST_PASS_DF['model_slug'] == spec.slug)
        & (FIRST_PASS_DF['mechanical_status'] != 'PASS')
    ]
    if rows.empty:
        continue
    print(f'\n===== {spec.label}: {len(rows)} non-PASS first-pass attempts =====')
    for row in rows.head(8).itertuples():
        attempt = next((a for a in ATTEMPTS if a['tag'] == row.tag), None)
        if attempt is None:
            continue
        print(f'--- {row.tag} [{row.mechanical_status} / {row.failed_stage}]')
        for error in attempt['verdict']['errors'][:3]:
            print('   ', error[:240])
    if len(rows) > 8:
        print(f'   ... and {len(rows) - 8} more (see attempts.csv)')


## J. Semantic review support — human review, not auto-scored

For each first-pass generation, the notebook writes one Google Drive review sheet containing:

- the exact repo situation shown to the model;
- players, state variables and actions;
- aggregate observations plus transition-level observation samples;
- terminal logic and reached terminal states;
- probability-related source inspection and probabilities actually observed by the validator;
- that repo case's own `should include` and `should NOT include` judging criteria.

The criteria are never shown during generation. These sheets support semantic review; they do not turn mechanical PASS into semantic success.

In [ ]:
def quote_markdown(text: str) -> str:
    return '\n'.join('> ' + line for line in text.splitlines())


def review_sheet(case: dict, attempt: dict) -> str:
    probe = attempt.get('probe') or {}
    structure = probe.get('structure') or {}
    sweep = probe.get('sweep') or {}
    verdict = attempt['verdict']
    inspection = attempt.get('inspection') or {}
    signals = ANALYSIS.structure_signals(attempt)

    lines = [
        f"# {case['case_id']} — {case['title']}",
        '',
        (
            f"*variant:* `{attempt['prompt_variant']}` · "
            f"*generation:* {attempt['generation']} · "
            f"*stage:* {attempt['stage']} · "
            f"*model:* {attempt.get('model_label') or attempt['meta'].get('model_slug')} "
            f"(`{attempt['meta'].get('model_id')}`)"
        ),
        (
            f"*mechanical:* **{verdict['mechanical_status']}**"
            + (
                '' if verdict['mechanical_status'] == 'PASS'
                else f" at `{verdict['failed_stage']}` — {(verdict['errors'] or [''])[0][:200]}"
            )
        ),
        f"*source case:* `{case['source_file']}` · *semantic:* `pending_human_review`",
        '',
        '## Situation (exact text given to the model)',
        '',
        quote_markdown(case['situation']),
        '',
    ]

    if not attempt['code_generated']:
        lines += [
            '## No model was produced', '',
            f"`{(verdict['errors'] or [''])[0]}`", ''
        ]
    else:
        lines += ['## Players', '']
        lines += [
            f'- `{player}` — {description}'
            for player, description in (structure.get('players') or {}).items()
        ] or ['- (none)']

        lines += ['', '## State variables', '']
        for name, metadata in (structure.get('state_variables') or {}).items():
            description = metadata.get('description', '') if isinstance(metadata, dict) else metadata
            lines.append(f'- `{name}` — {description}')

        lines += [
            '', f"Initial state: `{structure.get('initial_state')}`",
            '', '## Actions', ''
        ]
        for action, description in (structure.get('action_catalogue') or {}).items():
            lines.append(f'- `{action}` — {description}')

        lines += ['', 'Reachable actions per player:', '']
        for player, actions in (structure.get('actions_by_player') or {}).items():
            lines.append(f'- `{player}`: {actions}')
        if structure.get('unused_catalogue_actions'):
            lines.append(
                f"- declared but never legal: `{structure['unused_catalogue_actions']}`"
            )

        lines += ['', '## Aggregate observations per player', '']
        for player, observations in (structure.get('observations') or {}).items():
            lines.append(f'- `{player}`: {observations}')

        lines += ['', '## Transition-level observation samples', '']
        transition_samples = structure.get('transition_samples') or []
        if not transition_samples:
            lines.append('- no samples available')
        for sample in transition_samples:
            lines += [
                f"- state: `{sample.get('state')}`",
                f"  - joint action: `{sample.get('actions')}`",
                f"  - next state: `{sample.get('next_state')}`",
                f"  - probability: `{sample.get('probability')}`",
                f"  - observations: `{sample.get('observations')}`",
            ]

        lines += [
            '', '## Terminal logic', '', '```python',
            (inspection.get('is_terminal_source') or '(not found)').strip(),
            '```', '', 'Reached terminal states:', ''
        ]
        terminal_states = structure.get('terminal_states') or []
        lines += [f'- `{state}`' for state in terminal_states] or [
            '- none sampled; inspect mechanical status and caps'
        ]

        stochastic_observed = [
            value for value in (sweep.get('probabilities') or []) if 0 < value < 1
        ]
        lines += [
            '', '## Probability-related source inspection', '',
            f"- probability-like module assignments: `{inspection.get('probability_like_assignments') or 'none'}`",
            f"- fractional numeric literals inside `transition_kernel`: `{inspection.get('kernel_numeric_literals') or 'none'}`",
            f"- stochastic probabilities actually observed during sweep: `{stochastic_observed or 'none'}`",
            '',
            '*These are inspection aids, not claims that a probability is justified or unjustified.*',
            '', '## Mechanical size / exploration', '',
            f"- {signals['n_players']} players · {signals['n_state_vars']} state variables · {signals['n_actions']} actions",
            f"- {signals['n_states_explored']} states explored · max depth {signals['max_depth']} · {sweep.get('n_transition_entries')} transition entries",
            f"- caps hit: `{signals['caps_hit'] or 'none'}`",
            '',
        ]

    lines += [
        '## Reviewer checklist — repo `should include` (never shown to the model)', ''
    ]
    lines += [f'- [ ] {item}' for item in case['expect']]
    lines += ['', '## Reviewer checklist — repo `should NOT include`', '']
    lines += [f'- [ ] {item}' for item in case['avoid']]
    lines += [
        '', '## Reviewer notes', '',
        '_(Consider missing stakeholders, invented actors/details, unsupported probabilities, '
        'observability mistakes, arbitrary horizons, missing actions/state variables, and '
        'whether uncertainty was resolved by fiat.)_', ''
    ]
    return '\n'.join(lines)


for attempt in ATTEMPTS:
    if attempt['repair_round'] != 0:
        continue
    case = next(case for case in CASES if case['case_id'] == attempt['case_id'])
    (folder('review') / f"{attempt['tag']}.md").write_text(
        review_sheet(case, attempt), encoding='utf-8'
    )

print('review sheets written to:', folder('review'))

In [ ]:
# Display one first-pass review sheet inline, per model, so the reviewer can see how
# differently the models answer the same situation.
first_attempts = [a for a in ATTEMPTS if a['repair_round'] == 0]
seen = set()
for attempt in first_attempts:
    slug = attempt.get('model_slug')
    if slug in seen:
        continue
    seen.add(slug)
    display(Markdown((folder('review') / f"{attempt['tag']}.md").read_text()))
    if len(seen) >= 2:      # two is enough inline; the rest are on Drive
        break


In [ ]:
# One case, every model and generation side by side — useful when writing the discussion.
if len(FIRST_PASS_DF):
    comparison_case_id = CASES[0]['case_id']
    comparison = FIRST_PASS_DF[FIRST_PASS_DF['case_id'] == comparison_case_id][[
        'model_slug', 'generation', 'mechanical_status', 'failed_stage',
        'n_players', 'n_state_vars', 'n_actions', 'max_depth',
        'human_action_width_max', 'human_agency_share',
        'is_stochastic', 'n_distinct_probabilities', 'caps_hit',
    ]].sort_values(['model_slug', 'generation'])
    print(f'{comparison_case_id} across models:')
    display(comparison)

    print('\nMedian produced-model size by model (valid attempts only):')
    display(DERIVED['structure'].round(2))


## K. Saved artefacts, reproducibility, and open methodology questions

### Saved artefacts

Everything produced by this notebook is under the **Google Drive run directory**:

- `config.json`, `preflight_models.csv`, `provenance.json`, `manifest.tsv`
- `cost_probe.csv` — what the billing probe measured, if it ran
- `prompt_divergences.csv` — the four documented differences from the JSON track's prompt
- `posg_abc.py`, `probe.py`, `source_cases/*.md`
- `prompts/v2_python.system.md` — the exact system message, hashed in `provenance.json`
- `prompts/*.md`, `raw/*.txt`, `generated/*.py`, `probe/*.json`, `selftest/*`
- `records.jsonl` — one JSON record per attempt, the resume cache and the raw input to
  every number below
- `attempts.csv`, `first_pass.csv`, `csv/*.csv` — the tidy frames and every derived table
- `figures/*.png`, `figures/*.pdf` — the fourteen figures this design produces
- `tables/*.tex`, `tables/*.md` — booktabs tables, ready to `\input{}`
- `graphs/*.png` — per-model state graphs and the per-case model comparisons
- `eda/*.png`, `eda/eda_summary.csv`, `eda/eda_by_model.csv` — `world_model_eda` pages
  and statistics, on the same footing as the game track
- `v2/first_pass/*.json` — **the handoff.** Every mechanically valid first-pass model in
  the v2 schema `json_generation/lane1.py` consumes
- `v2/repaired/*.json` — the same for repaired models, kept separate on purpose
- `review/*.md` — one human-review sheet per first-pass generation

### Reproducing the analysis without re-generating

`records.jsonl` is sufficient. Offline:

```python
import json, pathlib
from python_prompting import analysis, figures, tables
attempts = [json.loads(line) for line in
            pathlib.Path('records.jsonl').read_text().splitlines() if line.strip()]
frame = analysis.attempts_frame(attempts)
first = analysis.first_pass(frame)
```

### Open questions deliberately not patched by assumption

1. **Players versus passive stakeholders:** the prompt asks for affected humans as
   players, while the ABC requires every player to have an action in every non-terminal
   state. A bystander who cannot act has nowhere to go but a `noop`, which is why the
   agency criterion reads the *maximum* action width and not the mean — a model is not
   credited or penalised for how often it follows our own NOOP instruction.
2. **Possible human goals:** the project is about humans' ability to reach many possible
   future goals, but neither this ABC nor the JSON schema has a goal-set field. The
   prompt therefore asks for zero rewards, and the power-estimation algorithms supply
   their own objective downstream.
3. **Unsupported probabilities:** introduced and observed numbers are recorded but never
   automatically judged as justified. A model that writes `0.5` for every uncertain event
   passes every mechanical check.
4. **Observations do not cross to the EDA:** `world_model_eda` describes dynamics, not
   observation structure, because the JSON schema it was built for has no observations.
   Partial observability is measured here instead by `has_hidden_state` and
   `observations_differ_by_player` (figure F13).
5. **One prompt means no prompt evidence.** This design can say what these models do
   under the team's shared prompt. It cannot say what a different prompt would buy —
   that was the ablation, and at this grid size it was about 4.5× underpowered
   (`analysis.design_sensitivity`). Trading it for comparability with the JSON track was
   a deliberate choice, and it is a limitation to state in the paper rather than a
   question the results answer.
6. **Semantic quality is not measured at all.** Everything machine-checked here is
   Lane 1. The review sheets are scaffolding for Lane 3, not Lane 3 itself.


In [ ]:
def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


PROVENANCE = {
    'source_repo_url': REPO_URL,
    'source_commit_sha': SOURCE_COMMIT_SHA,
    'source_branch': SOURCE_BRANCH,
    'source_clone_git_removed': not (REPO_ROOT / '.git').exists(),
    'posg_abc_path': 'posg.py',
    'posg_abc_sha256': sha256_bytes(POSG_SOURCE.encode('utf-8')),
    'probe_module_sha256': sha256_bytes(
        (REPO_ROOT / 'python_prompting' / 'posg_probe.py').read_bytes()),
    'n_v2_first_pass': len(list(folder('first_pass').glob('*.json'))),
    'n_v2_repaired': len(list(folder('repaired').glob('*.json'))),
    'case_files': {
        case['source_file']: sha256_bytes((REPO_ROOT / case['source_file']).read_bytes())
        for case in CASES
    },
    'registry': REGISTRY,
    'prompt_strategy': STRATEGY,
    'prompt_module_sha256': sha256_bytes(
        (REPO_ROOT / 'python_prompting' / 'prompts.py').read_bytes()),
    'system_prompt_sha256': sha256_bytes(SYSTEM_PROMPT.encode('utf-8')),
    'models': CONFIG['models'],
    'models_held_back': CONFIG['models_held_back'],
    'azure_billing_checked': AZURE_BILLING_CHECKED,
    'n_generations': N_GENERATIONS,
    'smoke_test_only': SMOKE_TEST_ONLY,
    'bootstrap_draws': BOOTSTRAP_DRAWS,
    'analysis_module_sha256': sha256_bytes(
        (REPO_ROOT / 'python_prompting' / 'analysis.py').read_bytes()),
    'figures_module_sha256': sha256_bytes(
        (REPO_ROOT / 'python_prompting' / 'figures.py').read_bytes()),
    'models_module_sha256': sha256_bytes(
        (REPO_ROOT / 'python_prompting' / 'models.py').read_bytes()),
    'graphs_module_sha256': sha256_bytes(
        (REPO_ROOT / 'python_prompting' / 'graphs.py').read_bytes()),
    'extraction_module_sha256': sha256_bytes(
        (REPO_ROOT / 'python_prompting' / 'extraction.py').read_bytes()),
    'run_layout': RUN_LAYOUT,
    'n_state_graphs': len(list(folder('graphs').glob('*.png'))),
    'n_eda_pages': len(list(folder('eda').glob('*.png'))),
    'eda_bridge_module_sha256': sha256_bytes(
        (REPO_ROOT / 'python_prompting' / 'eda_bridge.py').read_bytes()),
    'world_model_eda_sha256': sha256_bytes(
        (REPO_ROOT / 'world_model_eda' / 'eda.py').read_bytes()),
    'figures_written': sorted(name for name, paths in FIGURE_PATHS.items() if paths),
    'tables_written': sorted(TABLE_PATHS),
    'packages': {},
}

for package in ('openai', 'pandas', 'matplotlib', 'numpy', 'networkx', 'torch',
                'transformers', 'accelerate', 'bitsandbytes', 'huggingface_hub'):
    try:
        PROVENANCE['packages'][package] = importlib_metadata.version(package)
    except Exception:
        PROVENANCE['packages'][package] = None

(folder('info') / 'provenance.json').write_text(
    json.dumps(PROVENANCE, indent=2, default=str), encoding='utf-8'
)

# Final safety assertions.
if not str(RUN_DIR).startswith('/content/drive/'):
    raise AssertionError('RUN_DIR is not on Google Drive.')
if str(RUN_DIR).startswith(str(REPO_ROOT)):
    raise AssertionError('RUN_DIR is inside the source repository.')
if (REPO_ROOT / '.git').exists():
    raise AssertionError('Source clone unexpectedly contains .git.')

manifest_lines = []
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file():
        manifest_lines.append(f'{path.relative_to(RUN_DIR)}\t{path.stat().st_size}')

(folder('info') / 'manifest.tsv').write_text(
    '\n'.join(manifest_lines) + '\n', encoding='utf-8'
)

print('Drive run directory:', RUN_DIR)
print('source .git present:', (REPO_ROOT / '.git').exists())
print('all experiment outputs are under Drive:', str(RUN_DIR).startswith('/content/drive/'))
print('\nSaved files:')
for line in manifest_lines:
    print(' ', line)